# Predicting ICU trajectories and outcomes from the first 48 hours

**MD6117 Machine Learning for Healthcare AI: group project notebook**

Both tasks use the **PhysioNet/Computing in Cardiology Challenge 2012** data: 12,000 adult ICU stays of at least 48 hours, with irregularly sampled vitals and labs (`data/README.md` describes the dataset).

| | Task 1: SOFA trajectory forecasting | Task 2: In-hospital mortality prediction |
|---|---|---|
| **Question** | At hour *T* of the ICU stay, where will the patient's organ-failure (SOFA) score be 6 h from now? | At the 48 h landmark, will the patient die before hospital discharge? |
| **Prediction time** | *T* ∈ {24, 30, 36, 42} h | 48 h |
| **Inputs** | everything measured before *T* | everything measured in [0, 48) h |
| **Output** | six organ sub-scores (0–4) and their total; P(SOFA rises ≥ 2) | P(in-hospital death) |
| **Unit / n** | stay × decision time, 48,000 rows (12,000 stays) | stay, 11,833 rows |
| **Bar to beat** | persistence ("nothing changes") | bedside scores (SAPS-I, SOFA) and a ladder of simpler models |
| **Primary metric** | MAE (bootstrap 95 % CI) | AUPRC and AUROC (bootstrap 95 % CI) |

**How to run.** Run all cells top to bottom from the repository root; the `README.md` lists the environment. The three switches in the next cell control what gets recomputed. With the defaults, the notebook loads the frozen pre-processing outputs and the saved task 2 gradient-boosting models, and retrains everything else.

**Contents.** *Part A (Task 1):* A1 problem · A2 data & cleaning · A3 rule-based SOFA engine · A4 targets · A5 features · A6 split · A7 baselines · A8 methods · A9 training · A10 results · A11 interpretation · A12 discussion. *Part B (Task 2):* B1 problem · B2 pre-processing · B3 baseline ladder L0–L3 · B4 gradient boosting L4 · B5 test evaluation · B6 calibration & subgroups · B7 interpretation · B8 challenge metric · B9 discussion. *Part C:* summary and saved artefacts.

In [ ]:
# ---- run switches ---------------------------------------------------------------------
REBUILD_PREPROCESSING = False  # True: rebuild preprocessed/ from data/icu_records (task 1 ~2 min, task 2 ~8 min)
RETRAIN_TASK1 = True           # False: load trained_models/task1_sofa/ instead of refitting (CPU)
RETRAIN_TASK2_SEARCH = False   # True: rerun the 50-candidate x 5-fold CV searches for XGBoost and
                               #       CatBoost (GPU recommended; ~13 + ~89 min on the original GPU)

# ---- environment ------------------------------------------------------------------------
import os, sys, io, json, time, warnings, contextlib
from pathlib import Path
from functools import reduce

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import joblib
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.precision", 3)

import sklearn, xgboost, catboost, lightgbm, shap
print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__,
      "| catboost", catboost.__version__, "| lightgbm", lightgbm.__version__, "| shap", shap.__version__)
print("repo root:", ROOT)

In [ ]:
# ---- one plotting style for every figure ------------------------------------------------
# Validated categorical palette (light surface). Colour follows the entity, never its rank:
# each model family keeps its colour and marker in every chart of the notebook.
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
BASELINE = "#8a8984"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div_blue_red", ["#1c5cab", "#86b6ef", "#f0efec", "#f1a3a2", "#c23434"])

FAMILY_STYLE = {   # colour + marker: marker shape is the secondary (non-colour) encoding
    "persistence":     dict(color=BASELINE,  marker="s", linestyle="--"),
    "ordinal":         dict(color=SERIES[0], marker="o", linestyle="-"),
    "linear":          dict(color=SERIES[1], marker="v", linestyle="-"),
    "histgb":          dict(color=SERIES[2], marker="D", linestyle="-"),
    "catboost":        dict(color=SERIES[3], marker="^", linestyle="-"),
    "lightgbm":        dict(color=SERIES[4], marker="P", linestyle="-"),
    "histgb_absolute": dict(color=SERIES[6], marker="X", linestyle=":"),
}

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "lines.markersize": 6, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})

RESULTS = {"task1": ROOT / "results" / "task1_sofa", "task2": ROOT / "results" / "task2_mortality"}
for _d in RESULTS.values():
    (_d / "figures").mkdir(parents=True, exist_ok=True)


def savefig(fig, task, name):
    """Save a figure under results/<task>/figures/ and show it inline."""
    fig.savefig(RESULTS[task] / "figures" / f"{name}.png")
    plt.show()


def ci_str(point, lo, hi, nd=3):
    return f"{point:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

---
# Part A · Task 1: SOFA trajectory forecasting

## A1 Problem statement

SOFA (Sequential Organ Failure Assessment) scores six organ systems from 0 to 4: respiratory, coagulation, liver, cardiovascular, neurological and renal. Clinicians normally score it once every 24 h, looking back. This task asks whether routinely collected data can turn it into a short-horizon **forecast**.

* **Decision times.** *T* ∈ {24, 30, 36, 42} h after ICU admission.
* **Inputs.** Only measurements time-stamped before *T* (an expanding window).
* **Primary outcome.** The trailing-24 h SOFA over [*T*−18, *T*+6) h, i.e. the score **6 h ahead** of *T*: six sub-scores, summed into the total.
* **Secondary outcome.** `deteriorate_24h` = 1 when that score is ≥ 2 points above `SOFA_now`, the score over the last fully observed 24 h [*T*−24, *T*).
* **Outcome source.** Both outcomes come from our own rule engine (A3), never from `outcomes.csv`. That file is used only as an external reference check (D6).
* **Success criterion.** Beat persistence (`SOFA_hat = SOFA_now`) on total-SOFA MAE at ≥ 2 of the 4 horizons. For deterioration, beat the no-skill reference: AUROC 0.5, AUPRC = prevalence.
* **Known structural limit.** The dataset contains **no vasopressor data**, so the cardiovascular sub-score can only be 0 or 1. SOFA's levels 2–4 are defined by vasopressor dose.

Design decisions are labelled D1–D24 (decision log in `docs/task1_sofa/SOFA_forecast_project_plan.docx`). Section headers cross-reference TRIPOD+AI items.

In [ ]:
from src.task1_sofa import config as t1cfg, evaluate as t1eval
from src.task1_sofa.data_prep import parsing, qc as t1qc, scoring, splits as t1splits
from src.task1_sofa.data_prep import targets as t1targets, features as t1features
from src.task1_sofa import models as t1m

AXES = t1m.AXES          # ("resp", "coag", "liver", "cardio", "neuro", "renal")
AXIS_NAME = {"resp": "Respiratory", "coag": "Coagulation", "liver": "Liver",
             "cardio": "Cardiovascular", "neuro": "Neurological", "renal": "Renal"}
for T in t1cfg.HORIZONS_H:
    print(f"T={T:>2}h  features: t < {T}h   target window {tuple(x // 60 for x in t1cfg.target_window_min(T))}h"
          f"   SOFA_now window {tuple(x // 60 for x in t1cfg.now_window_min(T))}h")

## A2 Data source, cleaning and cohort *(TRIPOD+AI 5–7)*

All 12,000 records are used, with no exclusions beyond the dataset's own ≥ 48 h stay rule. Cleaning follows §7 of the methodology document (`docs/task1_sofa/基于PhysioNet…SOFA计算方法.docx`):

* drop the `-1` missing sentinel and rows with an empty `Parameter`;
* drop `MAP`/`NIMAP`/`PaO2`/`SaO2` ≤ 0 (waveform dropouts);
* drop `Weight` ≤ 20 kg, and `Height` < 100 or > 250 cm.

The raw-file audit below must reproduce the document's counts exactly.

In [ ]:
t0 = time.time()
if t1cfg.RECORDS_CLEAN.exists() and not REBUILD_PREPROCESSING:
    records = joblib.load(t1cfg.RECORDS_CLEAN); source = "cache (preprocessed/task1_sofa/records_clean.joblib)"
else:
    records = parsing.parse_all(); joblib.dump(records, t1cfg.RECORDS_CLEAN, compress=3); source = "raw files"
print(f"{len(records):,} cleaned records loaded from {source} in {time.time() - t0:.1f}s")

audit = t1qc.audit_raw()
audit_tbl = pd.DataFrame({"computed": audit["counts"], "methodology doc": t1qc.DOC_REF})
audit_tbl["match"] = audit_tbl["computed"] == audit_tbl["methodology doc"]
assert audit_tbl["match"].all(), "raw-data audit does not reproduce the methodology document"
audit_tbl

In [ ]:
desc = pd.DataFrame({rid: d for rid, (_s, d) in records.items()}).T
icutype = desc["ICUType"]
last_t = audit["last_t_hours"]
cohort = pd.Series({
    "stays": f"{len(desc):,}",
    "age, median [IQR]": f"{desc.Age.median():.0f} [{desc.Age.quantile(.25):.0f}–{desc.Age.quantile(.75):.0f}]",
    "male": f"{100 * (desc.Gender == 1).mean():.1f} %",
    "ICU type 1 / 2 / 3 / 4 (CCU / cardiac-surgery / medical / surgical)":
        " / ".join(f"{(icutype == k).sum():,}" for k in (1, 2, 3, 4)),
    "in-hospital mortality": f"{100 * pd.read_csv(t1cfg.OUTCOMES_CSV)['In-hospital_death'].mean():.1f} %",
    "last observation (h), median [min, p5]": f"{np.median(last_t):.1f} [{last_t.min():.1f}, {np.percentile(last_t, 5):.1f}]",
    "records ending before 42 h / 36 h": f"{int((last_t < 42).sum())} / {int((last_t < 36).sum())}",
}, name="value").to_frame()
cohort

## A3 The rule-based SOFA engine and its validation

The forecast target is computed by our own SOFA rule engine, `src/task1_sofa/data_prep/scoring.py`. It implements the methodology document's §3 table and §5 dataset rules. Within a window, each organ is scored from its worst value:

* **Respiratory:** min PaO₂/FiO₂, each PaO₂ paired with the nearest FiO₂ within ±2 h. Levels 3–4 require mechanical ventilation.
* **Coagulation:** min platelets.
* **Liver:** max bilirubin, carried forward from earlier windows if unmeasured.
* **Cardiovascular:** MAP < 70 mmHg → 1. Levels 2–4 are unreachable without vasopressor data.
* **Neurological:** min GCS.
* **Renal:** max creatinine, or the urine total when it covers ≥ 20 h.

A missing organ counts as 0 in the total (the "lenient" convention).

Before the engine defines anything we train on, we check it against two independent references:

1. the document's worked examples;
2. the official `outcomes.csv` SOFA, a first-24 h score computed by the challenge organisers *with* vasopressor data.

In [ ]:
DAY1, DAY2 = (0, 1440), (1440, 2880)
rows = []
for rid, (series, d) in records.items():
    s1, s2 = scoring.score_window(series, *DAY1), scoring.score_window(series, *DAY2)
    rows.append({"RecordID": rid, "ICUType": d.get("ICUType"), "SOFA_D1": s1["sofa_total"],
                 "SOFA_D2": s2["sofa_total"], **{f"{a}_d1": s1[a] for a in AXES},
                 **{f"{a}_d2": s2[a] for a in AXES}})
day = pd.DataFrame(rows).sort_values("RecordID").reset_index(drop=True)
day["SOFA_48h_max"] = day[["SOFA_D1", "SOFA_D2"]].max(axis=1)
day.to_csv(t1cfg.OUT_DIR / "sofa_day_scores.csv", index=False)

# worked examples from the methodology document (sections 5-6): (Day 1, Day 2, 48 h max)
for rid, expected in [(100001, (1, 2, 2)), (100005, (9, 9, 9))]:
    got = tuple(int(day.loc[day.RecordID == rid, c].iloc[0]) for c in ("SOFA_D1", "SOFA_D2", "SOFA_48h_max"))
    assert got == expected, (rid, got, expected)
    print(f"worked example {rid}: computed {got} = document {expected}")

oc = pd.read_csv(t1cfg.OUTCOMES_CSV)
m = day.merge(oc, on="RecordID")
m = m[m.SOFA != -1].copy()          # 403 stays have no reference SOFA


def agreement(pred, ref):
    d = pred - ref
    return {"n": len(d), "exact %": 100 * (d == 0).mean(), "within ±1 %": 100 * (d.abs() <= 1).mean(),
            "within ±2 %": 100 * (d.abs() <= 2).mean(), "MAE": d.abs().mean(),
            "bias (ours − reference)": d.mean(), "Pearson r": stats.pearsonr(pred, ref)[0]}


worse_d2 = m[m.SOFA_D2 > m.SOFA_D1]
agree_tbl = pd.DataFrame({
    "SOFA_D1 vs reference (all)": agreement(m.SOFA_D1, m.SOFA),
    "SOFA_48h_max vs reference (all)": agreement(m.SOFA_48h_max, m.SOFA),
    "SOFA_D1 vs reference (Day 2 worse)": agreement(worse_d2.SOFA_D1, worse_d2.SOFA),
    "SOFA_48h_max vs reference (Day 2 worse)": agreement(worse_d2.SOFA_48h_max, worse_d2.SOFA),
}).T
agree_tbl

In [ ]:
# Where does the mean bias come from? Compare the reference's implied cardiovascular
# sub-score (reference total minus our other five organs) with ours (capped at 0/1).
ours_cardio = m.cardio_d1.fillna(0)
implied_ref_cardio = (m.SOFA - (m.SOFA_D1 - ours_cardio)).mean()
print(f"average cardiovascular sub-score per patient: reference ~ {implied_ref_cardio:.2f}, ours {ours_cardio.mean():.2f}")
print(f"shortfall {ours_cardio.mean() - implied_ref_cardio:+.2f}  vs  mean(SOFA_D1 - reference) "
      f"{(m.SOFA_D1 - m.SOFA).mean():+.2f}  -> the cardiovascular cap explains the bias")

fig, ax = plt.subplots(figsize=(5.4, 4.6))
hi = int(max(m.SOFA_D1.max(), m.SOFA.max())) + 1
edges = np.arange(-0.5, hi + 1.5, 1)
H, _, _ = np.histogram2d(m.SOFA_D1, m.SOFA, bins=[edges, edges])
pcm = ax.pcolormesh(edges, edges, H.T, cmap=SEQ, norm=LogNorm(vmin=1, vmax=H.max()))
ax.plot([-0.5, hi + 0.5], [-0.5, hi + 0.5], color=INK2, lw=1, ls="--")
ax.set(xlabel="our rule-based SOFA, first 24 h (SOFA_D1)", ylabel="reference SOFA (outcomes.csv)",
       title="Rule engine vs reference SOFA (n = %s)" % f"{len(m):,}")
ax.grid(False)
fig.colorbar(pcm, ax=ax, label="stays (log scale)")
savefig(fig, "task1", "A3_rule_sofa_vs_reference")

**Reading A3.** The engine reproduces both worked examples exactly. Against the reference SOFA:

* **The reference is a first-24 h score, not the 48 h worst.** Among the stays whose Day 2 is worse, `SOFA_D1` tracks the reference better than `SOFA_48h_max` does.
* **The gap is one input.** Our score sits about 0.9 points *below* the reference on average. The cardiovascular accounting explains that bias: SOFA's cardiovascular levels 2–4 are defined by vasopressor dose, which this dataset does not contain, so our cardiovascular sub-score is capped at 0/1.
* **It is not a scoring error.** The correlation is high and almost all of the error is one-directional, which is the signature of a missing input.

We therefore use the engine as defined, state the cap as a limitation, and never use `outcomes.csv` as a training signal.

## A4 Forecast targets *(TRIPOD+AI 8)*

For each stay and decision time, `build_targets` scores two windows:

* the **target window** [*T*−18, *T*+6) h → six sub-scores and the total;
* the **now window** [*T*−24, *T*) h → `SOFA_now`.

From these come the deltas (target − now) and `deteriorate_24h`. An organ with no measurement in the target window keeps a missing target and is excluded from that organ's training and evaluation rows (D17); it is never zero-filled. Two consistency checks follow:

* the *T* = 42 h target window *is* Day 2, so its totals must equal A3's `SOFA_D2`;
* `SOFA_now` at *T* = 24 h *is* `SOFA_D1`, so it must reproduce A3's agreement with the reference.

In [ ]:
t0 = time.time()
if t1cfg.TARGETS_CSV.exists() and not REBUILD_PREPROCESSING:
    tg = pd.read_csv(t1cfg.TARGETS_CSV); source = "cache"
else:
    tg = t1targets.build_targets(records); tg.to_csv(t1cfg.TARGETS_CSV, index=False); source = "built"
print(f"targets {tg.shape} ({source}, {time.time() - t0:.1f}s); rows per stay: "
      f"{tg.groupby('RecordID').size().value_counts().to_dict()}")

chk = tg[tg.origin_h == 42].merge(day[["RecordID", "SOFA_D2"]], on="RecordID")
assert (chk.sofa_total.fillna(0) == chk.SOFA_D2).all()
ref = tg[tg.origin_h == 24].merge(oc[["RecordID", "SOFA"]], on="RecordID").query("SOFA != -1")
print(f"T=42 target == Day-2 SOFA for all {len(chk):,} stays;  SOFA_now(T=24) vs reference: "
      f"r = {stats.pearsonr(ref.sofa_now, ref.SOFA)[0]:.3f}, bias = {(ref.sofa_now - ref.SOFA).mean():+.3f}")

tg.groupby("origin_h")[["sofa_total", "sofa_now", "sofa_delta", "deteriorate_24h",
                        "cardio_instability", "target_window_empty"]].mean().round(3)

### How much is there to forecast? Target volatility by organ

Before fitting anything we ask how much each organ's score actually moves in 6 h. A model can only beat "nothing changes" where something *does* change, and it needs enough measured rows to learn from.

The table pools all splits and horizons. It describes the outcome variable itself, not a model, so pooling cannot leak anything.

In [ ]:
n_rows = len(tg)
vol = pd.DataFrame({
    AXIS_NAME[a]: {
        "rows with Δ defined": int(tg[f"{a}_delta"].notna().sum()),
        "% of rows measured": 100 * tg[f"{a}_score"].notna().mean(),
        "% unchanged in 6 h (Δ = 0)": 100 * (tg[f"{a}_delta"].dropna() == 0).mean(),
        "mean |Δ|": tg[f"{a}_delta"].abs().mean(),
        "std(Δ)": tg[f"{a}_delta"].std(),
    } for a in AXES}).T.sort_values("std(Δ)", ascending=False)
display(vol.round(3))

dist = pd.DataFrame({AXIS_NAME[a]: tg[f"{a}_score"].value_counts(normalize=True).reindex(range(5)).mul(100)
                     for a in ("cardio", "renal", "neuro")}).T
dist.columns = [f"% at {k}" for k in range(5)]
print("Score-value distribution (target window, measured rows). Cardiovascular can only be 0 or 1:")
display(dist.round(1).fillna("unreachable"))

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.scatter(vol["% of rows measured"], vol["std(Δ)"], s=70, color=SERIES[0], edgecolor=SURFACE, linewidth=2, zorder=3)
label_pos = {"Coagulation": ((-8, 4), "right"), "Cardiovascular": ((7, -9), "left")}   # avoid overlap
for name, r in vol.iterrows():
    xy, ha = label_pos.get(name, ((7, -3), "left"))
    ax.annotate(name, (r["% of rows measured"], r["std(Δ)"]), xytext=xy, textcoords="offset points",
                ha=ha, color=INK2, fontsize=9)
ax.set(xlabel="% of rows with the organ measured in the target window", ylabel="std of the true 6-h change (Δ)",
       title="Only the neurological score is both volatile and well measured", xlim=(30, 105))
savefig(fig, "task1", "A4_target_volatility")

## A5 Predictors *(TRIPOD+AI 9)* and missing data *(TRIPOD+AI 11)*

`build_features` summarises every input over [0, *T*) into four groups:

* `v_*`: for each of 36 time-varying variables, the last, first, min, max, mean, std, count, hours since last value, hours of coverage and 12 h slope;
* `e_*`: engineered haemodynamic, neurological, renal and respiratory signals;
* `sofa0_24_*`: rule-engine sub-scores for the admission day;
* `sofa_now_*`: sub-scores for the last 24 h, plus `d_*` admission descriptors.

No predictor uses data from *T* onward, and nothing comes from `outcomes.csv` (D6).

**Missing predictors** stay `NaN`. The tree models route missing values natively, so "never measured" is not silently turned into "measured normal". Only the linear family imputes, with medians fitted on train. Two exceptions:

* bilirubin is carried forward across windows;
* admission weight falls back to the cohort sex-specific median for the urine-rate feature only, flagged by `e_weight_imputed`.

In [ ]:
t0 = time.time()
if t1cfg.FEATURES_CSV.exists() and not REBUILD_PREPROCESSING:
    ft = pd.read_csv(t1cfg.FEATURES_CSV); source = "cache"
else:
    ft = t1features.build_features(records); ft.to_csv(t1cfg.FEATURES_CSV, index=False); source = "built"
fam = pd.Series([c.split("_")[0] for c in ft.columns if c not in ("RecordID", "origin_h")]).value_counts()
print(f"features {ft.shape} ({source}, {time.time() - t0:.1f}s); {ft.shape[1] - 2} predictor columns + origin_h: {fam.to_dict()}")

nan_frac = ft.drop(columns=["RecordID", "origin_h"]).isna().mean()
print(f"fully populated columns: {(nan_frac == 0).sum()} / {len(nan_frac)};  median missing fraction {nan_frac.median():.2f}")

# leakage / construction checks
ok = True
for v in ["v_HR_count", "v_GCS_count", "v_Urine_count", "v_Creatinine_count"]:
    piv = ft.pivot_table(index="RecordID", columns="origin_h", values=v)
    ok &= bool(((piv[24] <= piv[30]) & (piv[30] <= piv[36]) & (piv[36] <= piv[42])).all())
assert ok, "measurement counts must be non-decreasing in T (expanding window)"
assert (ft.pivot_table(index="RecordID", columns="origin_h", values="sofa0_24_total").nunique(axis=1) == 1).all()
assert not [c for c in ft.columns if c.lower() in ("sofa", "saps-i", "survival", "length_of_stay", "in-hospital_death")]
print("checks passed: expanding-window counts monotone in T; admission-day SOFA identical across T; no outcomes.csv column")

## A6 Data partitioning *(TRIPOD+AI 12a)*

The split is at record level, so a stay's four horizon rows never straddle splits. It is 70/15/15 train/validation/test, stratified on SOFA band × in-hospital death, with seed 42, and frozen. Hyperparameters are chosen on **validation**; **test** is scored once, after selection.

In [ ]:
sp = t1splits.make_splits()
sp.to_csv(t1cfg.SPLITS_CSV, index=False)
print(sp.split.value_counts().to_dict())
t1splits.split_summary(sp, icutype)

## A7 Baselines *(TRIPOD+AI 12e)*

Before any model is fitted, we fix the bar each task must clear (D18):

* **Regression: persistence**, `SOFA_hat(T+6) = SOFA_now`. A model that cannot beat it is only relearning "the future looks like the present".
* **Classification: the no-skill reference.** AUROC = 0.5, but AUPRC = **prevalence**, not 0.5 (Saito & Rehmsmeier 2015). With roughly 4 % positives, an AUPRC of 0.2 can sound good while still being weak.

Both are computed on the training split here; validation and test values appear with the results.

In [ ]:
J = ft.merge(tg, on=["RecordID", "origin_h"]).merge(sp[["RecordID", "split"]], on="RecordID")
base = J.dropna(subset=["sofa_total"])
print("rows with a total-SOFA target, per split x horizon:")
display(base.groupby(["split", "origin_h"]).size().unstack())

persist = t1eval.regression_report(base[base.split == "train"], "sofa_total", {"persistence": "sofa_now"})
print("total-SOFA persistence baseline (train):")
display(persist.round(3))

no_skill = pd.DataFrame([{"origin_h": T, **t1eval.no_skill_reference(
    base[(base.split == "train") & (base.origin_h == T)].deteriorate_24h)} for T in t1cfg.HORIZONS_H])
print("deterioration no-skill reference (train):")
display(no_skill.round(4))

## A8 Model development methods *(TRIPOD+AI 12b–12g, 13–15)*

We compare **five model families head-to-head** under one protocol, plus one controlled ablation. Each organ gets its own model, and the six predictions are summed into the total.

| Family | What it predicts | Why it is in the comparison |
|---|---|---|
| **Ordinal** (D19; Frank & Hall 2001) | absolute score, via 4 binary classifiers P(score > *k*), *k* = 0…3. E[score] = Σₖ P(score > *k*) | The score is ordinal (0–4), not continuous. |
| **Linear** (D20) | Δ from `SOFA_now` (Ridge, plus class-weighted logistic regression for deterioration); median-impute + standardise | Does non-linearity earn its keep? |
| **HistGB**, *primary* (D12–D14, D17) | Δ from `SOFA_now`; one pooled model per organ across all four horizons, with `origin_h` as a feature | Native missing-value handling, no extra dependency. |
| **CatBoost** (D21) | Δ; ICU type, sex and `origin_h` as native categoricals; ordered boosting | Less target-leakage overfitting on small organs. |
| **LightGBM** (D22) | Δ; leaf-wise tree growth | A structurally different gradient-boosting bias. |
| *HistGB-absolute* (ablation) | absolute score, otherwise identical to HistGB | Separates "absolute target" from "ordinal decomposition" (A10). |

**Protocol, identical for every family:**

* Each family has a fixed four-point grid. Every point is fit on train and scored on validation.
* We never use a library's internal early stopping, which would silently reuse training rows.
* Regressors are selected on validation MAE; classifiers on validation AUPRC.
* Δ predictions are reconstructed as `clip(SOFA_now + Δ, 0, 4)`.
* For every family, an organ with no `SOFA_now` anchor predicts 0 (D24), so all families are scored on the identical population.

In [ ]:
VARIED = ("learning_rate", "max_leaf_nodes", "depth", "num_leaves", "alpha", "C")


def _fmt(v):
    try:
        return f"{float(v):g}"
    except (TypeError, ValueError):
        return str(v)


def cfg_label(params):
    return ", ".join(f"{k}={_fmt(params[k])}" for k in VARIED if k in params)


grids = {"ordinal (per threshold)": t1m.ordinal.GRID, "linear Ridge": t1m.linear.RIDGE_GRID,
         "linear logistic": t1m.linear.LOGISTIC_GRID, "histgb": t1m.REGRESSOR_GRID,
         "catboost": t1m.catboost_model.REGRESSOR_GRID, "lightgbm": t1m.lightgbm_model.REGRESSOR_GRID}
pd.DataFrame({name: [cfg_label(p) for p in g] + [""] * (4 - len(g)) for name, g in grids.items()},
             index=[f"config {i + 1}" for i in range(4)]).T

## A9 Model training *(TRIPOD+AI 12c)*

With `RETRAIN_TASK1 = True`, each family is fitted here and saved to `trained_models/task1_sofa/<family>/`. Each family's chosen hyperparameters and every grid point's validation score go to `model_spec.json`. With `False`, the saved models are loaded instead, and any family missing from disk is still trained.

Each table shows the validation score of every grid point per organ. ★ marks the selected configuration.

In [ ]:
SPEC_PATH = t1cfg.MODELS_DIR / "model_spec.json"
spec = json.loads(SPEC_PATH.read_text()) if SPEC_PATH.exists() else {}
spec.update({"seed": t1cfg.SEED, "horizons": list(t1cfg.HORIZONS_H),
             "feature_columns": t1m.feature_columns(ft)})
spec.setdefault("families", {})


def _records(grid):
    if isinstance(grid, pd.DataFrame):
        return grid.to_dict("records")
    return {str(k): _records(v) for k, v in grid.items()}


def save_family(name, axis_models, axis_params, axis_grids, det=None, det_params=None, det_grid=None):
    d = t1cfg.MODELS_DIR / name
    d.mkdir(parents=True, exist_ok=True)
    for axis, mdl in axis_models.items():
        joblib.dump(mdl, d / f"axis_{axis}.joblib")
    entry = {"axis_params": axis_params, "validation_grids": _records(axis_grids)}
    if det is not None:
        joblib.dump(det, d / "deteriorate_24h.joblib")
        entry.update(deterioration_params=det_params, deterioration_grid=_records(det_grid))
    spec["families"][name] = entry
    SPEC_PATH.write_text(json.dumps(spec, indent=2, default=str))


def load_family(name):
    d = t1cfg.MODELS_DIR / name
    axis_models = {a: joblib.load(d / f"axis_{a}.joblib") for a in AXES}
    det = joblib.load(d / "deteriorate_24h.joblib") if (d / "deteriorate_24h.joblib").exists() else None
    return axis_models, det


def needs_training(name):
    return RETRAIN_TASK1 or not (t1cfg.MODELS_DIR / name / "axis_resp.joblib").exists()


def grid_table(grids, metric, best="min"):
    """rows = organ, columns = grid configuration, values = validation metric; ★ = selected."""
    out = {}
    for axis, g in grids.items():
        g = pd.DataFrame(g)
        vals = dict(zip(g.apply(cfg_label, axis=1), g[metric]))
        pick = min(vals, key=vals.get) if best == "min" else max(vals, key=vals.get)
        out[AXIS_NAME.get(axis, axis)] = {k: f"{v:.4f}{' ★' if k == pick else ''}" for k, v in vals.items()}
    return pd.DataFrame(out).T


def show_family(name, metric="val_mae", best="min"):
    entry = spec["families"].get(name, {})
    if "validation_grids" in entry:
        display(grid_table(entry["validation_grids"], metric, best))
    else:
        display(pd.DataFrame(entry.get("axis_params", {})).T)


train_times = {}

### A9.1 Ordinal (D19)

In [ ]:
t0 = time.time()
if needs_training("ordinal"):
    ordinal_models, ordinal_params, ordinal_grids = t1m.ordinal.train_axis_models_ordinal(ft, tg, sp)
    save_family("ordinal", ordinal_models, ordinal_params, ordinal_grids)
else:
    ordinal_models, _ = load_family("ordinal")
train_times["ordinal"] = time.time() - t0
print(f"ordinal: 6 organs x 4 thresholds ({train_times['ordinal'] / 60:.1f} min). Validation AUROC per threshold classifier:")
og = spec["families"]["ordinal"].get("validation_grids")
if og:
    display(pd.concat({AXIS_NAME[a]: grid_table({f"P(score>{k})": g for k, g in og[a].items()},
                                                "val_auroc", "max") for a in AXES}))

### A9.2 Linear baseline (D20)

In [ ]:
t0 = time.time()
if needs_training("linear"):
    linear_models, linear_params, linear_grids = t1m.linear.train_axis_models_linear(ft, tg, sp)
    det_linear, det_params_linear, det_grid_linear = t1m.linear.train_deterioration_model_linear(ft, tg, sp)
    save_family("linear", linear_models, linear_params, linear_grids, det_linear, det_params_linear, det_grid_linear)
else:
    linear_models, det_linear = load_family("linear")
train_times["linear"] = time.time() - t0
print(f"linear ({train_times['linear'] / 60:.1f} min). Ridge, validation MAE of the delta:")
show_family("linear")
print("deterioration (logistic), validation AUPRC:", spec["families"]["linear"]["deterioration_params"])

### A9.3 HistGB, the primary family (D12–D14, D17)

In [ ]:
t0 = time.time()
if needs_training("histgb"):
    histgb_models, histgb_params, histgb_grids = t1m.train_axis_models(ft, tg, sp)
    det_histgb, det_params_histgb, det_grid_histgb = t1m.train_deterioration_model(ft, tg, sp)
    save_family("histgb", histgb_models, histgb_params, histgb_grids, det_histgb, det_params_histgb, det_grid_histgb)
else:
    histgb_models, det_histgb = load_family("histgb")
train_times["histgb"] = time.time() - t0
print(f"histgb ({train_times['histgb'] / 60:.1f} min). Validation MAE of the delta:")
show_family("histgb")
dg = spec["families"]["histgb"].get("deterioration_grid")
if dg:
    print("deterioration classifier, validation AUPRC:")
    display(grid_table({"deteriorate_24h": dg}, "val_auprc", "max"))

### A9.4 CatBoost (D21) and LightGBM (D22)

In [ ]:
t0 = time.time()
if needs_training("catboost"):
    catboost_models, catboost_params, catboost_grids = t1m.catboost_model.train_axis_models_catboost(ft, tg, sp)
    det_catboost, det_params_cb, det_grid_cb = t1m.catboost_model.train_deterioration_model_catboost(ft, tg, sp)
    save_family("catboost", catboost_models, catboost_params, catboost_grids, det_catboost, det_params_cb, det_grid_cb)
else:
    catboost_models, det_catboost = load_family("catboost")
train_times["catboost"] = time.time() - t0
print(f"catboost ({train_times['catboost'] / 60:.1f} min). Validation MAE of the delta:")
show_family("catboost")

t0 = time.time()
if needs_training("lightgbm"):
    lightgbm_models, lightgbm_params, lightgbm_grids = t1m.lightgbm_model.train_axis_models_lightgbm(ft, tg, sp)
    det_lightgbm, det_params_lgb, det_grid_lgb = t1m.lightgbm_model.train_deterioration_model_lightgbm(ft, tg, sp)
    save_family("lightgbm", lightgbm_models, lightgbm_params, lightgbm_grids, det_lightgbm, det_params_lgb, det_grid_lgb)
else:
    lightgbm_models, det_lightgbm = load_family("lightgbm")
train_times["lightgbm"] = time.time() - t0
print(f"lightgbm ({train_times['lightgbm'] / 60:.1f} min). Validation MAE of the delta:")
show_family("lightgbm")

### A9.5 Ablation: HistGB predicting the absolute score

Ordinal differs from HistGB in **two** ways at once: it predicts the absolute score rather than the Δ, *and* it uses the cumulative-binary decomposition. This control keeps HistGB's learner, grid and protocol and changes only the target column (`{organ}_score` instead of `{organ}_delta`). The comparison in A10 can then tell the two effects apart.

In [ ]:
t0 = time.time()
if needs_training("histgb_absolute"):
    absolute_models, absolute_params, absolute_grids = t1m.histgb_absolute.train_axis_models_absolute(ft, tg, sp)
    save_family("histgb_absolute", absolute_models, absolute_params, absolute_grids)
else:
    absolute_models, _ = load_family("histgb_absolute")
train_times["histgb_absolute"] = time.time() - t0
print(f"histgb_absolute ({train_times['histgb_absolute'] / 60:.1f} min). Validation MAE of the absolute score:")
show_family("histgb_absolute")
print("training time (min):", {k: round(v / 60, 1) for k, v in train_times.items()})

## A10 Results *(TRIPOD+AI 20–23)*

Every family is predicted on all 48,000 rows and joined with the targets. We report validation, used for selection, and **test**, opened once, with 1,000-resample bootstrap 95 % CIs. Persistence uses the same 0-fallback for a missing `SOFA_now` that every model uses (D24), so all series are scored on the same rows.

In [ ]:
pred_frames = [t1m.ordinal.predict_axis_scores_ordinal(ordinal_models, ft),
               t1m.linear.predict_axis_scores_linear(linear_models, ft),
               t1m.predict_axis_scores(histgb_models, ft),
               t1m.catboost_model.predict_axis_scores_catboost(catboost_models, ft),
               t1m.lightgbm_model.predict_axis_scores_lightgbm(lightgbm_models, ft),
               t1m.histgb_absolute.predict_axis_scores_absolute(absolute_models, ft)]
preds = reduce(lambda a, b: a.merge(b, on=["RecordID", "origin_h"]), pred_frames)
full = (preds.merge(ft, on=["RecordID", "origin_h"])
             .merge(sp[["RecordID", "split"]], on="RecordID")
             .merge(tg, on=["RecordID", "origin_h"]))
for a in AXES:
    full[f"persistence_{a}"] = full[f"sofa_now_{a}"].fillna(0.0)

FAMILIES = ["ordinal", "linear", "histgb", "catboost", "lightgbm"]
TOTAL_COLS = {"persistence": "sofa_now", "ordinal": "sofa_total_pred_ordinal", "linear": "sofa_total_pred_linear",
              "histgb": "sofa_total_pred", "catboost": "sofa_total_pred_catboost",
              "lightgbm": "sofa_total_pred_lightgbm", "histgb_absolute": "sofa_total_pred_abs"}


def axis_cols(a):
    return {"persistence": f"persistence_{a}", "ordinal": f"{a}_pred_ordinal", "linear": f"{a}_pred_linear",
            "histgb": f"{a}_pred", "catboost": f"{a}_pred_catboost", "lightgbm": f"{a}_pred_lightgbm",
            "histgb_absolute": f"{a}_pred_abs"}


total_rep = {s: t1eval.regression_report(full[full.split == s].dropna(subset=["sofa_total"]), "sofa_total", TOTAL_COLS)
             for s in ("val", "test")}
for s, rep in total_rep.items():
    rep.to_csv(RESULTS["task1"] / f"total_sofa_mae_{s}.csv", index=False)


def mae_pivot(rep, series):
    t = rep.assign(cell=lambda d: [ci_str(*r) for r in d[["mae", "mae_lo", "mae_hi"]].to_numpy()])
    return t.pivot(index="origin_h", columns="series", values="cell")[series]


headline = ["persistence"] + FAMILIES
print("TOTAL-SOFA MAE [95% CI], validation:")
display(mae_pivot(total_rep["val"], headline))
print("TOTAL-SOFA MAE [95% CI], TEST:")
display(mae_pivot(total_rep["test"], headline))

In [ ]:
rep = total_rep["test"]
fig, axes_ = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.35, 1]})
ax = axes_[0]
for s in headline:
    d = rep[rep.series == s]
    ax.plot(d.origin_h, d.mae, label=s, **FAMILY_STYLE[s])
ax.set(xlabel="decision time T (h)", ylabel="total-SOFA MAE (test)", xticks=list(t1cfg.HORIZONS_H),
       title="Total-SOFA forecast error by horizon")
ax.legend(ncol=2, fontsize=9)

ax = axes_[1]
mean_mae = rep.groupby("series").mae.mean().reindex(headline)
ypos = np.arange(len(headline))[::-1]
ax.barh(ypos, mean_mae.values, color=[FAMILY_STYLE[s]["color"] for s in headline], height=0.6, edgecolor=SURFACE, linewidth=2)
for y, v in zip(ypos, mean_mae.values):
    ax.text(v + 0.01, y, f"{v:.3f}", va="center", fontsize=9, color=INK2)
ax.set(yticks=ypos, yticklabels=headline, xlabel="mean MAE over the 4 horizons (test)", title="Average over horizons",
       xlim=(0, mean_mae.max() * 1.18))
ax.grid(axis="y", visible=False)
fig.tight_layout()
savefig(fig, "task1", "A10_total_sofa_mae_test")

beats = {s: int((rep[rep.series == s].set_index("origin_h").mae <
                 rep[rep.series == "persistence"].set_index("origin_h").mae).sum()) for s in FAMILIES}
print("horizons (of 4) where each family beats persistence on test MAE:", beats)

### Per-organ error: where does the gain come from?

Next, MAE per organ, pooled over the four horizons, on test. The heat map shows each family's MAE **relative to persistence**: below 1 (blue) means better than "nothing changes", above 1 (red) means worse.

In [ ]:
test_rows = full[full.split == "test"]
per_axis = {}
for a in AXES:
    d = test_rows.dropna(subset=[f"{a}_score"])
    per_axis[AXIS_NAME[a]] = {s: t1eval._mae(d[f"{a}_score"], d[c]) for s, c in axis_cols(a).items()}
per_axis = pd.DataFrame(per_axis).T[["persistence"] + FAMILIES + ["histgb_absolute"]]
per_axis.to_csv(RESULTS["task1"] / "per_axis_mae_test.csv")
print("Per-organ MAE, test, pooled over horizons:")
display(per_axis.round(3))

by_h = pd.concat({AXIS_NAME[a]: t1eval.regression_report(test_rows.dropna(subset=[f"{a}_score"]), f"{a}_score", axis_cols(a))
                  for a in AXES}, names=["axis"]).reset_index(level=0)
by_h.to_csv(RESULTS["task1"] / "per_axis_mae_by_horizon_test.csv", index=False)

ratio = per_axis[FAMILIES + ["histgb_absolute"]].div(per_axis["persistence"], axis=0)
fig, ax = plt.subplots(figsize=(7.6, 3.9))
lim = float(np.abs(np.log2(ratio.values)).max())
im = ax.imshow(np.log2(ratio.values), cmap=DIV, vmin=-lim, vmax=lim, aspect="auto")
for i in range(ratio.shape[0]):
    for j in range(ratio.shape[1]):
        ax.text(j, i, f"{ratio.values[i, j]:.2f}", ha="center", va="center", fontsize=9, color=INK)
ax.set(xticks=range(ratio.shape[1]), xticklabels=[c.replace("_", "\n") for c in ratio.columns],
       yticks=range(ratio.shape[0]), yticklabels=ratio.index,
       title="Per-organ test MAE relative to persistence (<1 better, >1 worse)")
ax.grid(False)
cb = fig.colorbar(im, ax=ax, ticks=np.log2([0.5, 0.75, 1, 1.33, 2]))
cb.ax.set_yticklabels(["0.5", "0.75", "1", "1.33", "2"])
savefig(fig, "task1", "A10_per_axis_mae_ratio_test")

### Ablation: absolute target vs ordinal decomposition

If predicting the *absolute* score were what makes ordinal best, HistGB-absolute should close the gap to ordinal. The delta-accuracy table then asks whether the models track real change, rather than just exploiting how rarely SOFA moves. On test, for total SOFA, the predicted Δ is the predicted total minus `SOFA_now`. Directional accuracy is computed over the rows where the true total changes.

In [ ]:
abl = total_rep["test"].pivot(index="origin_h", columns="series", values="mae")[
    ["persistence", "histgb", "histgb_absolute", "ordinal"]]
abl.loc["mean"] = abl.mean()
print("Total-SOFA test MAE: HistGB + delta vs HistGB + absolute vs ordinal + absolute")
display(abl.round(3))

d = test_rows.dropna(subset=["sofa_total"])
true_delta = d.sofa_total - d.sofa_now
moved = true_delta != 0
delta_tbl = {}
for s in ["ordinal", "histgb", "histgb_absolute", "persistence"]:
    pd_ = d[TOTAL_COLS[s]] - d.sofa_now
    delta_tbl[s] = {"n": len(d), "delta MAE": (pd_ - true_delta).abs().mean(),
                    "directional accuracy (rows that change)": (np.sign(pd_[moved]) == np.sign(true_delta[moved])).mean(),
                    "corr(pred Δ, true Δ)": np.corrcoef(pd_, true_delta)[0, 1] if pd_.std() > 0 else np.nan}
print(f"Delta-specific accuracy, total SOFA, test ({int(moved.sum()):,} of {len(d):,} rows change):")
pd.DataFrame(delta_tbl).T.round(3)

### Deterioration classifier (≥ 2-point rise)

Ordinal has no classifier variant, so four families are compared here. The ROC and PR curves are on test; the dashed line is the no-skill reference.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score, brier_score_loss

det_frames = [t1m.predict_deterioration(det_histgb, ft),
              t1m.linear.predict_deterioration_linear(det_linear, ft),
              t1m.catboost_model.predict_deterioration_catboost(det_catboost, ft),
              t1m.lightgbm_model.predict_deterioration_lightgbm(det_lightgbm, ft)]
det = (reduce(lambda a, b: a.merge(b, on=["RecordID", "origin_h"]), det_frames)
       .merge(sp[["RecordID", "split"]], on="RecordID")
       .merge(tg[["RecordID", "origin_h", "deteriorate_24h"]], on=["RecordID", "origin_h"]).dropna())
PROB_COLS = {"histgb": "deteriorate_proba", "linear": "deteriorate_proba_linear",
             "catboost": "deteriorate_proba_catboost", "lightgbm": "deteriorate_proba_lightgbm"}
det_rep = {}
for s in ("val", "test"):
    dd = det[det.split == s]
    det_rep[s] = t1eval.classification_comparison(dd, "deteriorate_24h", PROB_COLS)
    ns = t1eval.no_skill_reference(dd.deteriorate_24h)
    det_rep[s] = pd.concat([det_rep[s], pd.DataFrame([{"series": "no-skill", "n": len(dd), "auroc": 0.5,
                                                         "auprc": ns["auprc"], "brier": ns["brier"]}])])
    det_rep[s].to_csv(RESULTS["task1"] / f"deterioration_{s}.csv", index=False)
    print(f"--- {s}: deterioration classifiers (prevalence {ns['prevalence']:.4f}) ---")
    display(det_rep[s].round(3))

dt = det[det.split == "test"]
fig, axes_ = plt.subplots(1, 2, figsize=(10.5, 4.3))
for s, col in PROB_COLS.items():
    st = FAMILY_STYLE[s]
    fpr, tpr, _ = roc_curve(dt.deteriorate_24h, dt[col])
    prec, rec, _ = precision_recall_curve(dt.deteriorate_24h, dt[col])
    axes_[0].plot(fpr, tpr, color=st["color"], label=f"{s} (AUROC {roc_auc_score(dt.deteriorate_24h, dt[col]):.3f})")
    axes_[1].plot(rec, prec, color=st["color"], label=f"{s} (AUPRC {average_precision_score(dt.deteriorate_24h, dt[col]):.3f})")
axes_[0].plot([0, 1], [0, 1], color=BASELINE, ls="--", lw=1, label="no skill")
axes_[1].axhline(dt.deteriorate_24h.mean(), color=BASELINE, ls="--", lw=1, label="no skill (prevalence)")
axes_[0].set(xlabel="false-positive rate", ylabel="true-positive rate", title="ROC, test")
axes_[1].set(xlabel="recall", ylabel="precision", title="Precision-recall, test")
for ax in axes_:
    ax.legend(fontsize=8.5)
fig.tight_layout()
savefig(fig, "task1", "A10_deterioration_roc_pr_test")

In [ ]:
cal = t1eval.calibration_table(dt.deteriorate_24h, dt.deteriorate_proba)
print("HistGB deterioration classifier, calibration by decile of predicted risk (test):")
display(cal.round(3))

cap = t1eval.capture_rate_report(dt.deteriorate_24h.to_numpy(), dt.deteriorate_proba.to_numpy())
print("Risk stratification: share of true deteriorations caught by flagging the top-k% of rows (test):")
display(cap.round(3))

fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.plot([0, 0.8], [0, 0.8], color=BASELINE, ls="--", lw=1, label="perfect calibration")
ax.plot(cal.mean_pred, cal.observed_rate, color=FAMILY_STYLE["histgb"]["color"], marker="D", label="histgb (deciles)")
ax.set(xlabel="mean predicted probability", ylabel="observed deterioration rate",
       title="Calibration, deterioration (test)", xlim=(0, 0.8), ylim=(0, 0.8))
ax.legend(fontsize=9)
savefig(fig, "task1", "A10_deterioration_calibration_test")

print("Subgroups, total-SOFA MAE (histgb):")
sub = pd.concat({
    "haemodynamic instability (validation)": t1eval.subgroup_report(
        full[full.split == "val"].dropna(subset=["sofa_total"]), "sofa_total", "sofa_total_pred", "cardio_instability"
    ).rename(columns={"cardio_instability": "group"}),
    "ICU type (test)": t1eval.subgroup_report(
        test_rows.dropna(subset=["sofa_total"]).merge(icutype.rename("ICUType"), left_on="RecordID", right_index=True),
        "sofa_total", "sofa_total_pred", "ICUType").rename(columns={"ICUType": "group"}),
})
display(sub.round(3))

## A11 Interpretation: permutation importance (D16)

Permutation importance for the primary family (HistGB), on a fixed 2,000-row validation sample with 3 repeats: how much the validation error grows when one feature is shuffled. HistGB has no built-in importances, and sampling keeps the run time tractable (D16).

In [ ]:
cols = t1m.feature_columns(ft)
val_rows = full[full.split == "val"]
imp_rows = []
for a, mdl in histgb_models.items():
    dd = val_rows.dropna(subset=[f"{a}_delta"])
    imp = t1eval.permutation_report(mdl, dd[cols], dd[f"{a}_delta"], top_k=5)
    imp_rows += [{"organ": AXIS_NAME[a], "rank": i + 1, "feature": r.feature, "importance": r.importance_mean}
                 for i, r in imp.iterrows()]
print("Top-5 features per organ model (increase in validation error when shuffled):")
display(pd.DataFrame(imp_rows).pivot(index="rank", columns="organ", values="feature")[[AXIS_NAME[a] for a in AXES]])

dv = det[det.split == "val"]
Xd = ft.set_index(["RecordID", "origin_h"]).loc[dv.set_index(["RecordID", "origin_h"]).index].reset_index()
imp_det = t1eval.permutation_report(det_histgb, Xd[cols], dv.deteriorate_24h.to_numpy(),
                                    scoring="average_precision", top_k=12)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ypos = np.arange(len(imp_det))[::-1]
ax.barh(ypos, imp_det.importance_mean, xerr=imp_det.importance_std, color=FAMILY_STYLE["histgb"]["color"],
        height=0.6, edgecolor=SURFACE, linewidth=2, error_kw=dict(ecolor=INK2, lw=1))
ax.set(yticks=ypos, yticklabels=imp_det.feature, xlabel="drop in validation AUPRC when shuffled",
       title="Deterioration classifier: top features")
ax.grid(axis="y", visible=False)
savefig(fig, "task1", "A11_deterioration_permutation_importance")

## A12 Discussion *(TRIPOD+AI 25–27)*

**Verdict on the four questions this task set out to answer** (all numbers are test-split values from A10, unless marked):

| Question | Verdict | Evidence |
|---|---|---|
| **H1:** Does ML add information beyond persistence for total SOFA? | **Supported** | Ordinal, HistGB, CatBoost and LightGBM beat persistence at all 4 horizons, on validation and on test. Mean test MAE: ordinal 0.591, HistGB 0.608, persistence 0.726, which meets the success criterion. The gain is largest at *T* = 24 h (0.63 vs 0.96, CIs well separated). By *T* = 36–42 h persistence is already good, and the CIs overlap. |
| **H2:** Is the gain uniform across organs? | **No: concentrated in one organ** | Only the **neurological** model beats persistence (pooled MAE 0.230 vs 0.308). For respiratory, coagulation, liver, cardiovascular and renal, every family is 7–90 % *worse* than "nothing changes". |
| **H3:** Does formulation matter beyond non-linearity? | **Supported** | The linear baseline loses to persistence at 3 of 4 horizons, so non-linearity matters. HistGB-*absolute* (0.618) is worse than HistGB-*delta* (0.608), yet ordinal, which also predicts the absolute score, is best (0.591). So ordinal's edge comes from the cumulative-binary decomposition, not from the absolute target. It also tracks real change: 84 % directional accuracy on rows whose SOFA moves, and correlation 0.70 with the true Δ. |
| **H4:** Does the same information detect deterioration (≥ 2-point rise)? | **Discrimination yes, calibration no** | AUROC 0.78–0.84 and AUPRC 0.18–0.31, against a prevalence of 0.045 (4–7× no-skill). Flagging the riskiest 10 % of checkpoints catches 48.8 % of deteriorations (lift 10.8×). But the top risk decile predicts 0.70 and observes 0.22. |

**Why only the neurological organ?** The volatility analysis (A4) answers this from the outcome itself, before any model is involved. Forecasting beats persistence only where the score both *moves* and is *measured*:

* **Neurological:** the only organ with both, std(Δ) 0.76 and 98 % measured.
* **Respiratory:** moves almost as much (0.57) but is measured in only 47 % of rows.
* **Renal, coagulation, cardiovascular:** measured in > 95 % of rows, but unchanged in ≥ 92 % of 6-h windows, so there is little to add to "no change".
* **Liver:** fails both conditions.

The cardiovascular organ has an extra, data-imposed ceiling: it can only take the values 0 and 1, so no model can extract more than a binary distinction. Permutation importance (A11) gives the same picture from the model side:

* The coagulation, liver, cardiovascular and renal models lean almost entirely on the current sub-score and the latest raw value. They have, in effect, learned a sophisticated persistence.
* The neurological model draws on several GCS summaries: last, mean, std and first value. That is genuine trajectory information.

**Subgroups.** Error does not differ by haemodynamic instability (MAE 0.592 in both groups). It is highest in cardiac-surgery recovery (0.74 vs 0.53–0.59 elsewhere). A plausible reason: these patients are typically sedated and ventilated after surgery and then wake quickly, so their GCS and respiratory scores swing more within the first 48 h.

**Limitations.**

1. **No vasopressor data.** The cardiovascular score is capped at 0/1. This also explains the whole 0.86-point mean gap to the reference SOFA (A3).
2. **The target comes from our own rule engine.** It reproduces the methodology document exactly but is not the clinical gold standard.
3. **Internal validation only.** One dataset, no external validation, and no calendar dates, so temporal drift cannot be assessed.
4. **Probabilities are miscalibrated.** Deterioration probabilities are inflated by `class_weight="balanced"` and need Platt or isotonic recalibration on validation before being read as risks.
5. **A deliberate scoring rule.** Organs without a current-state anchor predict 0 (D24). This keeps families comparable but gives up predictions from indirect evidence.
6. **Sampled interpretation.** Permutation importance uses a 2,000-row sample, and the ordinal threshold classifiers are not constrained to be monotone.

---
# Part B · Task 2: In-hospital mortality prediction at 48 h

## B1 Problem statement

At the **48-hour landmark** of an ICU stay, predict `In-hospital_death`: death before discharge from hospital, including deaths after leaving the ICU.

* **Inputs.** Observations with 0 ≤ `Time` < 48:00.
* **Cohort.** The 12,000 stays minus **167 with a negative length of stay**, all survivors. That leaves 11,833 stays, of whom 1,707 died (14.4 %).
* **Metrics.** **AUPRC** (primary; the no-skill AUPRC equals the prevalence) and **AUROC**, each with 1,000-resample stratified bootstrap 95 % CIs. For comparability with the 2012 challenge we also report min(Se, PPV).
* **Operating point.** The threshold is chosen on **validation** for recall ≈ 0.80, because a missed death costs more than an extra review, then applied to test.
* **Protocol.** We use a **ladder** of increasingly complex models: L0 random → L1 single variable → L2 bedside scores (SAPS-I, SOFA) → L3 eight-feature logistic regression → L4 gradient-boosted trees on all 1,474 features. Each rung must beat the one below to justify its complexity.
* **One test look.** **Test is scored once per declared model.** Every hyperparameter, threshold and feature choice uses train + validation only.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

from src.task2_mortality import preprocess as t2pre, verify_outputs as t2verify
from src.task2_mortality import validate_phase3_outputs as t2validate, data as t2data
from src.task2_mortality.metrics import compute_metrics, bootstrap_ci as strat_boot, find_threshold_for_recall, min_se_ppv

T2_PRE = ROOT / "preprocessed" / "task2_mortality"
T2_MODELS = ROOT / "trained_models" / "task2_mortality"
T2_RES = RESULTS["task2"]

## B2 Pre-processing *(TRIPOD+AI 7, 9, 11)*

`src/task2_mortality/preprocess.py` turns each stay into a single row. Its rules live in `config.json`:

* **Time window.** Rows timed exactly **48:00** contradict the dataset's 47:59 upper bound. They are excluded from the primary features and exported to a separate sensitivity file (232 stays).
* **Missing values.** `-1` means missing. Values outside clinical **physiological ranges** are set to missing, not clipped. Two unit errors are corrected first: pH > 14 is ÷100, and Height < 100 cm is ×100. Side effect: the age range (18–120) also blanks the 19 stays aged 15–17, so their age is imputed.
* **Same-minute duplicates.** Values recorded in the same minute are collapsed: continuous values by median, MechVent by max, urine by sum.
* **Features.** For 37 dynamic variables, three windows (0–24 h, 24–48 h, 0–48 h) × 12 statistics: measured, count, first, last, min, max, mean, median, std, slope per hour, delta and time span, plus urine total. Added to these: merged invasive/non-invasive blood-pressure streams, an arterial-line flag, static descriptors including BMI with missing-indicators, and record-level counters. **1,474 features** in total.
* **Excluded from X.** SAPS-I, SOFA, length of stay and survival. SAPS-I and SOFA are exported separately, as *scores* for rung L2.
* **Imputation.** The tree models get the **unimputed** matrix. The linear models get **train-median imputed** matrices (fill values fitted on train only). No scaling or feature selection happens here; both are model steps fitted inside train.

The outputs are frozen (2026-09-21, SHA-256 manifest). The independent verifiers below re-derive and check them.

In [ ]:
if REBUILD_PREPROCESSING or not (T2_PRE / "patient_features_unimputed.csv.gz").exists():
    t2pre.main()          # ~8 min; rewrites preprocessed/task2_mortality/
summary = json.loads((T2_PRE / "preprocessing_summary.json").read_text(encoding="utf-8"))

verification = t2verify.main()                 # raises AssertionError on any failed check
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    n_checks = t2validate.main()               # raises RuntimeError if any check fails
print(f"independent verification: {verification['status']} ({verification['verified_output_hashes']} output hashes, "
      f"{verification['verified_imputed_cells']:,} imputed cells re-derived); phase-3 validation: {n_checks}/{n_checks} checks passed")

a = summary["audits"]
audit2 = pd.Series({
    "raw rows read": f"{a['raw_rows']:,}",
    "stays excluded (negative length of stay)": 12000 - summary["source"]["record_files"],
    "stays kept / deaths": f"{summary['source']['record_files']:,} / {summary['source']['death_count']:,} "
                           f"({100 * summary['source']['death_prevalence']:.2f} %)",
    "missing-sentinel or out-of-range values": f"{a['sentinel_rows']:,}",
    "rows with empty Parameter": f"{a['empty_parameter_rows']:,}",
    "rows exactly at 48:00 (sensitivity file only)": f"{a['exact_48h_rows']:,} in {summary['output']['boundary_sensitivity_rows']} stays",
    "same-minute conflicting (time, parameter) keys": f"{a['conflicting_time_parameter_keys']:,}",
    "model features": summary["output"]["model_feature_columns"],
    "unimputed missing cells": f"{summary['output']['unimputed_missing_cells']:,}",
}, name="value").to_frame()
display(audit2)
display(pd.DataFrame(summary["split_summary"]).round(4))

In [ ]:
labels, X_unimp, X_imp = t2data.load_data()
FEATS = t2data.get_feature_names()
lab = labels.set_index("RecordID")


def split_data(split):
    """(RecordIDs, unimputed X, imputed X, y), all in labels-file order."""
    ids = labels.loc[labels.split == split, "RecordID"].to_numpy()
    Xu = X_unimp.set_index("RecordID").loc[ids, FEATS]
    Xi = X_imp[split].set_index("RecordID").loc[ids, FEATS]
    return ids, Xu, Xi, lab.loc[ids, "In-hospital_death"].to_numpy()


ids_tr, Xu_tr, Xi_tr, y_tr = split_data("train")
ids_va, Xu_va, Xi_va, y_va = split_data("validation")
ids_te, Xu_te, Xi_te, y_te = split_data("test")
print(f"train {len(y_tr):,} ({y_tr.sum()} deaths) | validation {len(y_va):,} ({y_va.sum()}) | test {len(y_te):,} ({y_te.sum()})")


def feature_group(c):
    if c.startswith("static_"): return "static descriptors (incl. BMI, missing flags)"
    if c.startswith("record__"): return "record-level counters"
    if c.endswith("has_arterial_line"): return "arterial-line indicator"
    if c.startswith("bp_"): return "merged blood pressure streams"
    return "dynamic parameters x windows x statistics"


display(pd.Series([feature_group(c) for c in FEATS]).value_counts().rename("features").to_frame())

fdict = pd.read_csv(T2_PRE / "feature_dictionary.csv")
meas = fdict[(fdict.statistic == "measured") & (fdict.window == "0_48h") & ~fdict.feature_name.str.startswith("bp_")]
cov = pd.Series({r.source_parameter: X_unimp[r.feature_name].mean() * 100 for r in meas.itertuples()}).sort_values()
fig, ax = plt.subplots(figsize=(6.4, 7.6))
ax.barh(cov.index, cov.values, color=SERIES[0], height=0.65, edgecolor=SURFACE, linewidth=1.5)
for y, v in enumerate(cov.values):
    ax.text(v + 1, y, f"{v:.0f}", va="center", fontsize=8, color=INK2)
ax.set(xlabel="% of stays with at least one measurement in 0-48 h", xlim=(0, 110),
       title="Measurement coverage by parameter (n = 11,833)")
ax.grid(axis="y", visible=False)
savefig(fig, "task2", "B2_measurement_coverage")

**Missingness carries information.** Lab tests are ordered when clinicians are worried, so *whether* something was measured (`measured`, `count`, `time_span_hours`) is kept as a feature rather than imputed away. TroponinI and cholesterol, for example, are measured in only a small minority of stays.

## B3 Baseline ladder L0–L3

Each rung has its threshold picked on validation (recall ≈ 0.80) and is scored on test once.

In [ ]:
ladder = []


def score_rung(rung, model, n_feat, s_val, s_test, y_val=None, y_test=None, n_boot=1000):
    """Test AUROC/AUPRC with stratified-bootstrap CI + Se/PPV/Sp at the validation recall-0.80 threshold."""
    y_val = y_va if y_val is None else y_val
    y_test = y_te if y_test is None else y_test
    thr = find_threshold_for_recall(y_val, s_val, target_recall=0.80)
    m = compute_metrics(y_test, s_test, threshold=thr)
    auc = strat_boot(y_test, s_test, roc_auc_score, n_bootstrap=n_boot)
    ap = strat_boot(y_test, s_test, average_precision_score, n_bootstrap=n_boot)
    row = {"rung": rung, "model": model, "features": n_feat, "n": len(y_test),
           "AUROC": auc[0], "AUROC lo": auc[1], "AUROC hi": auc[2], "AUPRC": ap[0], "AUPRC lo": ap[1], "AUPRC hi": ap[2],
           "threshold": thr, "Se": m["sensitivity"], "PPV": m["ppv"], "Sp": m["specificity"], "min(Se,PPV)": m["min_se_ppv"]}
    ladder.append(row)
    return row


def show(rows):
    t = pd.DataFrame(rows if isinstance(rows, list) else [rows])
    t["AUROC [95% CI]"] = [ci_str(*r) for r in t[["AUROC", "AUROC lo", "AUROC hi"]].to_numpy()]
    t["AUPRC [95% CI]"] = [ci_str(*r) for r in t[["AUPRC", "AUPRC lo", "AUPRC hi"]].to_numpy()]
    return t[["rung", "model", "features", "n", "AUROC [95% CI]", "AUPRC [95% CI]", "Se", "PPV", "Sp", "min(Se,PPV)"]].round(3)


# L0: random scores (no-skill floor)
rng = np.random.RandomState(42)
p_rand_te, p_rand_va = rng.uniform(size=len(y_te)), rng.uniform(size=len(y_va))
r0 = score_rung("L0", "random", 0, p_rand_va, p_rand_te)

# L1: one variable at a time, oriented so a higher score means higher risk
L1 = {"age": ("static_age_years", +1), "GCS min (0-48 h)": ("gcs__0_48h__min", -1),
      "BUN max (0-48 h)": ("bun__0_48h__max", +1)}
r1 = [score_rung("L1", name, 1, sign * Xi_va[col].to_numpy(), sign * Xi_te[col].to_numpy())
      for name, (col, sign) in L1.items()]
print(f"test prevalence (= no-skill AUPRC): {y_te.mean():.4f}")
show([r0] + r1)

### L2: bedside severity scores (SAPS-I, SOFA)

In `outcomes.csv`, SAPS-I and SOFA use **`-1` for "not computable"**. Kept as a number, `-1` becomes the *lowest* score, but these stays are actually **higher** risk. So each score is evaluated on the stays where it exists, and later models are compared with it on that same subset.

In [ ]:
te_lab = lab.loc[ids_te]
va_lab = lab.loc[ids_va]
sentinel = {}
for score, key in (("SAPS-I", "saps_i"), ("SOFA", "sofa")):
    avail = te_lab[f"{key}_available"] == 1
    raw = te_lab[key].fillna(-1)
    s_med = te_lab[key].fillna(te_lab.loc[avail, key].median())
    sentinel[score] = {
        "test stays without a score": int((~avail).sum()),
        "mortality: without score / with score": f"{te_lab.loc[~avail, 'In-hospital_death'].mean():.3f} / "
                                                 f"{te_lab.loc[avail, 'In-hospital_death'].mean():.3f}",
        "AUROC keep -1 as a number": roc_auc_score(te_lab["In-hospital_death"], raw),
        "AUROC replace -1 by median": roc_auc_score(te_lab["In-hospital_death"], s_med),
        "AUROC score-available stays only": roc_auc_score(te_lab.loc[avail, "In-hospital_death"], te_lab.loc[avail, key]),
    }
print("Effect of the -1 sentinel (test):")
display(pd.DataFrame(sentinel).T)

r2 = []
for score, key in (("SAPS-I", "saps_i"), ("SOFA", "sofa")):
    mv, mt = va_lab[f"{key}_available"] == 1, te_lab[f"{key}_available"] == 1
    r2.append(score_rung("L2", score, "score", va_lab.loc[mv, key].to_numpy(), te_lab.loc[mt, key].to_numpy(),
                         y_val=y_va[mv.to_numpy()], y_test=y_te[mt.to_numpy()]))
show(r2)

### L3: eight-feature logistic regression (clinical prior)

These eight features were chosen on clinical grounds, not by statistical screening. They cover the SOFA organ domains, the strongest L1 variables, and one measurement-behaviour signal: whether lactate was ordered at all.

The recipe: L2-regularised logistic regression (C = 1, balanced class weights), with `StandardScaler` fitted on train, on the train-median imputed matrices.

In [ ]:
L3_FEATURES = {
    "bun__0_48h__max": "Renal: BUN peak (best L1 variable)",
    "static_age_years": "Demographic: physiological reserve",
    "gcs__0_48h__min": "CNS: worst GCS (as SOFA neuro uses)",
    "urine__0_48h__min": "Renal: worst oliguria",
    "lactate__0_48h__max": "Perfusion: peak lactate (shock)",
    "platelets__0_48h__min": "Coagulation: lowest platelets",
    "creatinine__0_48h__max": "Renal: creatinine peak (AKI)",
    "lactate__0_48h__measured": "Measurement behaviour: lactate ordered",
}


def fit_logistic(features):
    sc = StandardScaler().fit(Xi_tr[features])
    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(sc.transform(Xi_tr[features]), y_tr)
    return lr, (lambda X: lr.predict_proba(sc.transform(X[features]))[:, 1])


l3_model, l3_predict = fit_logistic(list(L3_FEATURES))
coef = pd.DataFrame({"rationale": L3_FEATURES.values(), "coefficient (per SD)": l3_model.coef_[0]}, index=L3_FEATURES.keys())
display(coef.round(3))
fit_auc = {s: roc_auc_score(y, l3_predict(X)) for s, X, y in (("train", Xi_tr, y_tr), ("validation", Xi_va, y_va))}
print("train / validation AUROC:", {k: round(v, 4) for k, v in fit_auc.items()}, " -> small gap, no overfitting")
r3 = score_rung("L3", "logistic, 8 clinical-prior features", 8, l3_predict(Xi_va), l3_predict(Xi_te))
show(r3)

**Two coefficients have the "wrong" sign:** creatinine and platelets. Both are collinear with BUN and carry weak signal of their own. They are left as they are, because changing the feature set after seeing test would be a second look.

### Which *summary statistic* of a variable? GCS min vs GCS last (train + validation only)

Gradient-boosting SHAP (B7) later ranked `gcs__0_48h__last` first by a wide margin, while L3 had used the *minimum* GCS. Before touching test, we screen every GCS statistic on train + validation. Then we refit L3 with the swap, scoring validation only.

In [ ]:
y_tv = np.concatenate([y_tr, y_va])
gcs_screen = {}
for stat in ["last", "median", "mean", "max", "first", "slope_per_hour", "delta", "std", "min"]:
    col = f"gcs__0_48h__{stat}"
    x = np.concatenate([Xi_tr[col], Xi_va[col]])
    auc = roc_auc_score(y_tv, x)
    gcs_screen[stat] = {"AUROC (oriented)": max(auc, 1 - auc), "direction": "lower = riskier" if auc < 0.5 else "higher = riskier"}
print("Single-variable AUROC of each GCS statistic, train + validation:")
display(pd.DataFrame(gcs_screen).T)

orig = list(L3_FEATURES)
swap = {"gcs__0_48h__min": "gcs__0_48h__last", "urine__0_48h__min": "urine__0_48h__total"}
variants = {
    "A original (GCS min, urine min)": orig,
    "B GCS min -> last": [swap.get(f, f) if f.startswith("gcs") else f for f in orig],
    "C add GCS last, keep min": orig + ["gcs__0_48h__last"],
    "D urine min -> total": [swap.get(f, f) if f.startswith("urine") else f for f in orig],
    "E both swaps": [swap.get(f, f) for f in orig],
}
abl_rows = {}
for name, fs in variants.items():
    _, pr = fit_logistic(fs)
    abl_rows[name] = {"features": len(fs), "train AUROC": roc_auc_score(y_tr, pr(Xi_tr)),
                      "validation AUROC": roc_auc_score(y_va, pr(Xi_va)), "validation AUPRC": average_precision_score(y_va, pr(Xi_va))}
pd.DataFrame(abl_rows).T.round(4)

**L3-revised (pre-declared).** Variant E was selected on validation only. It is declared as a new rung, *L3-revised*, with its eight features fixed before test was scored, and is scored on test once. The original L3 stays on the ladder; it is not replaced.

In [ ]:
L3R_FEATURES = variants["E both swaps"]
l3r_model, l3r_predict = fit_logistic(L3R_FEATURES)
r3r = score_rung("L3-revised", "logistic, 8 features (GCS last, urine total)", 8, l3r_predict(Xi_va), l3r_predict(Xi_te))
show([r3, r3r])

## B4 Gradient-boosted trees on all 1,474 features (L4)

Two implementations, both on the **unimputed** matrix with native NaN handling and class-imbalance weighting (no resampling):

* **XGBoost:** `scale_pos_weight` = negatives/positives. A `RandomizedSearchCV` draws 50 candidates, scored by 5-fold stratified CV AUROC on train.
* **CatBoost:** `auto_class_weights="Balanced"`. The same 50 × 5-fold random search, as a resumable manual loop, then a final fit with early stopping on validation.

The searches ran on a 24 GB GPU (13 min for XGBoost, 89 min for CatBoost); their logs are in `results/task2_mortality/`. LightGBM was attempted but had no GPU support on that machine, so it produced no result. Set `RETRAIN_TASK2_SEARCH = True` to rerun both searches and overwrite the saved models.

In [ ]:
XGB_PATH, CB_PATH = T2_MODELS / "L4_xgboost_best.json", T2_MODELS / "L4_catboost_best.cbm"
XGB_SPACE = {"max_depth": [3, 5, 7, 9], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300, 500],
             "min_child_weight": [1, 3, 5], "subsample": [0.8, 0.9, 1.0], "colsample_bytree": [0.8, 0.9, 1.0],
             "gamma": [0, 0.1, 0.2], "max_bin": [63, 127, 255]}
CB_SPACE = {"depth": [4, 6, 8], "learning_rate": [0.01, 0.03, 0.05, 0.1], "iterations": [200, 300, 500],
            "l2_leaf_reg": [1, 3, 5, 9], "border_count": [32, 64, 128], "random_strength": [0.5, 1, 3],
            "bagging_temperature": [0, 0.5, 1]}
SPW = (len(y_tr) - y_tr.sum()) / y_tr.sum()


def _gpu_ok_xgb():
    try:
        xgb.XGBClassifier(device="cuda", n_estimators=1).fit(Xu_tr.to_numpy()[:100], y_tr[:100]); return True
    except Exception:
        return False


def run_xgb_search():
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
    gpu = _gpu_ok_xgb(); t0 = time.time()
    base = dict(objective="binary:logistic", eval_metric="auc", scale_pos_weight=SPW, random_state=42, n_jobs=-1,
                device="cuda" if gpu else "cpu", tree_method="hist")
    search = RandomizedSearchCV(xgb.XGBClassifier(**base), XGB_SPACE, n_iter=50, scoring="roc_auc", n_jobs=1,
                                cv=StratifiedKFold(5, shuffle=True, random_state=42), random_state=42, verbose=1)
    search.fit(Xu_tr.to_numpy(), y_tr)
    best = xgb.XGBClassifier(**base, **search.best_params_)
    best.fit(Xu_tr.to_numpy(), y_tr, eval_set=[(Xu_va.to_numpy(), y_va)], verbose=False)
    best.save_model(str(XGB_PATH))
    p = best.predict_proba(Xu_va.to_numpy())[:, 1]
    log = {"model": "xgboost", "gpu_used": gpu, "best_hyperparameters": search.best_params_,
           "cv_best_auroc": float(search.best_score_), "validation_auroc": float(roc_auc_score(y_va, p)),
           "validation_auprc": float(average_precision_score(y_va, p)), "training_time_seconds": time.time() - t0}
    (T2_RES / "L4_xgboost_training.json").write_text(json.dumps(log, indent=2))


def run_catboost_search():
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.utils import check_random_state
    t0 = time.time()
    base = dict(loss_function="Logloss", eval_metric="AUC", auto_class_weights="Balanced", random_seed=42,
                verbose=False, allow_writing_files=False, max_ctr_complexity=2)
    try:
        CatBoostClassifier(iterations=10, task_type="GPU", devices="0", **base).fit(Xu_tr.to_numpy()[:100], y_tr[:100])
        base.update(task_type="GPU", devices="0"); gpu = True
    except Exception:
        gpu = False
    ckpt_path = T2_RES / "L4_catboost_search_checkpoint.json"
    done = json.loads(ckpt_path.read_text())["results"] if ckpt_path.exists() else []
    rs = check_random_state(42)
    candidates = [{k: rs.choice(v).item() for k, v in CB_SPACE.items()} for _ in range(50)]
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    for i, cand in enumerate(candidates[len(done):], start=len(done) + 1):
        s = cross_val_score(CatBoostClassifier(**base, **cand), Xu_tr.to_numpy(), y_tr, cv=cv, scoring="roc_auc", n_jobs=1)
        done.append({"iteration": i, "params": cand, "mean_cv_auroc": float(s.mean()), "std_cv_auroc": float(s.std())})
        ckpt_path.write_text(json.dumps({"results": done}, indent=2))
    best = max(done, key=lambda r: r["mean_cv_auroc"])
    model = CatBoostClassifier(**{**base, **best["params"], "iterations": 2000})
    model.fit(Pool(Xu_tr.to_numpy(), y_tr), eval_set=Pool(Xu_va.to_numpy(), y_va), early_stopping_rounds=50, verbose=False)
    model.save_model(str(CB_PATH))
    p = model.predict_proba(Xu_va.to_numpy())[:, 1]
    log = {"model": "catboost", "gpu_used": gpu, "best_hyperparameters": best["params"],
           "cv_best_auroc": best["mean_cv_auroc"], "cv_best_auroc_std": best["std_cv_auroc"],
           "validation_auroc": float(roc_auc_score(y_va, p)), "validation_auprc": float(average_precision_score(y_va, p)),
           "training_time_seconds": time.time() - t0}
    (T2_RES / "L4_catboost_training.json").write_text(json.dumps(log, indent=2))


if RETRAIN_TASK2_SEARCH:
    run_xgb_search()
    run_catboost_search()

xgb_log = json.loads((T2_RES / "L4_xgboost_training.json").read_text())
cb_log = json.loads((T2_RES / "L4_catboost_training.json").read_text())
xgb_model = xgb.XGBClassifier(); xgb_model.load_model(str(XGB_PATH))
cb_model = CatBoostClassifier(); cb_model.load_model(str(CB_PATH))

cb_search = pd.DataFrame(json.loads((T2_RES / "L4_catboost_search_checkpoint.json").read_text())["results"])
print("CatBoost search, top 5 of 50 candidates by 5-fold CV AUROC:")
display(pd.concat([cb_search.params.apply(pd.Series), cb_search[["mean_cv_auroc", "std_cv_auroc"]]], axis=1)
        .sort_values("mean_cv_auroc", ascending=False).head(5).round(4))
print("selected XGBoost hyperparameters:", xgb_log["best_hyperparameters"])

In [ ]:
p_xgb = {s: xgb_model.predict_proba(X.to_numpy())[:, 1] for s, X in (("train", Xu_tr), ("validation", Xu_va))}
p_cb = {s: cb_model.predict_proba(X.to_numpy())[:, 1] for s, X in (("train", Xu_tr), ("validation", Xu_va))}
sel = pd.DataFrame({
    "XGBoost": {"CV AUROC (train, 5-fold)": xgb_log["cv_best_auroc"], "validation AUROC": roc_auc_score(y_va, p_xgb["validation"]),
                "validation AUPRC": average_precision_score(y_va, p_xgb["validation"]),
                "search time (min, GPU)": xgb_log["training_time_seconds"] / 60},
    "CatBoost": {"CV AUROC (train, 5-fold)": cb_log["cv_best_auroc"], "validation AUROC": roc_auc_score(y_va, p_cb["validation"]),
                 "validation AUPRC": average_precision_score(y_va, p_cb["validation"]),
                 "search time (min, GPU)": cb_log["training_time_seconds"] / 60}}).T
SELECTED = sel["validation AUROC"].idxmax()
print(f"model selection on validation AUROC -> {SELECTED}")
display(sel.round(4))

cap_rows = {}
for name, p, n_trees in (("XGBoost", p_xgb, xgb_model.get_booster().num_boosted_rounds()),
                         ("CatBoost", p_cb, cb_model.tree_count_)):
    cap_rows[name] = {"trees used": n_trees, "train AUROC": roc_auc_score(y_tr, p["train"]),
                      "validation AUROC": roc_auc_score(y_va, p["validation"]),
                      "train separated perfectly": bool(p["train"][y_tr == 1].min() > p["train"][y_tr == 0].max())}
cap_rows["L3 logistic (reference)"] = {"trees used": "-", "train AUROC": fit_auc["train"], "validation AUROC": fit_auc["validation"],
                                       "train separated perfectly": False}
cap = pd.DataFrame(cap_rows).T
cap["gap"] = cap["train AUROC"].astype(float) - cap["validation AUROC"].astype(float)
print("Capacity check (train vs validation; test untouched):")
display(cap)

**Capacity, not degraded generalisation.** XGBoost fits the training set exactly. That alone is not overfitting in the harmful sense: its validation AUROC holds up, and CatBoost, which early-stops and does *not* interpolate, shows a similar train–validation gap.

Two CPU-only ablations follow, using train and validation only:

1. **Early stopping.** Does adding early stopping to XGBoost's final fit change anything?
2. **Feature count.** How many of the 1,474 features does the model actually need? Features are ranked by mean |SHAP| on **train**, so validation is not used for selection, and XGBoost is refitted on the top *k*.

In [ ]:
best_hp = {k: v for k, v in xgb_log["best_hyperparameters"].items() if k != "n_estimators"}
cpu_base = dict(objective="binary:logistic", eval_metric="auc", scale_pos_weight=SPW, random_state=42,
                n_jobs=-1, device="cpu", tree_method="hist", **best_hp)
Xtr_np, Xva_np = Xu_tr.to_numpy(), Xu_va.to_numpy()

es_rows = {}
for name, extra in (("no early stopping, 300 trees", dict(n_estimators=300)),
                    ("early stopping (50 rounds), cap 2000", dict(n_estimators=2000, early_stopping_rounds=50))):
    t0 = time.time()
    mdl = xgb.XGBClassifier(**cpu_base, **extra).fit(Xtr_np, y_tr, eval_set=[(Xva_np, y_va)], verbose=False)
    es_rows[name] = {"trees": mdl.get_booster().num_boosted_rounds(), "train AUROC": roc_auc_score(y_tr, mdl.predict_proba(Xtr_np)[:, 1]),
                     "validation AUROC": roc_auc_score(y_va, mdl.predict_proba(Xva_np)[:, 1]), "seconds": time.time() - t0}
print("Early-stopping ablation (CPU refit with the selected hyperparameters):")
display(pd.DataFrame(es_rows).T.round(4))

contrib_tr = xgb_model.get_booster().predict(xgb.DMatrix(Xtr_np), pred_contribs=True)[:, :-1]
order = np.argsort(-np.abs(contrib_tr).mean(0))
fc_rows = []
for k in [8, 20, 50, 100, 200, 500, len(FEATS)]:
    idx = order[:k]
    mdl = xgb.XGBClassifier(**{**cpu_base, "n_estimators": 2000, "early_stopping_rounds": 50})
    mdl.fit(Xtr_np[:, idx], y_tr, eval_set=[(Xva_np[:, idx], y_va)], verbose=False)
    pv = mdl.predict_proba(Xva_np[:, idx])[:, 1]
    fc_rows.append({"k": k, "trees": mdl.get_booster().num_boosted_rounds(),
                    "validation AUROC": roc_auc_score(y_va, pv), "validation AUPRC": average_precision_score(y_va, pv)})
fc = pd.DataFrame(fc_rows)
fc["share of full-model AUROC %"] = 100 * fc["validation AUROC"] / fc["validation AUROC"].iloc[-1]
display(fc.round(4))

l3r_val_auc = roc_auc_score(y_va, l3r_predict(Xi_va))
fig, ax = plt.subplots(figsize=(6.2, 4))
ax.plot(fc.k, fc["validation AUROC"], color=SERIES[0], marker="o", label="XGBoost on top-k features")
ax.axhline(l3r_val_auc, color=SERIES[1], ls="--", lw=1.5, label=f"L3-revised logistic, 8 features ({l3r_val_auc:.3f})")
ax.set(xscale="log", xlabel="number of features (ranked by train-set SHAP)", ylabel="validation AUROC",
       title="How many features does L4 need?")
ax.set_xticks(fc.k); ax.set_xticklabels(fc.k)
ax.legend(fontsize=9, loc="upper left")
savefig(fig, "task2", "B4_feature_count_ablation")

## B5 Test evaluation of L4: the single test look

The selected model (validation AUROC) is scored on test once. Its threshold comes from validation (recall ≈ 0.80).

In [ ]:
final_model = xgb_model if SELECTED == "XGBoost" else cb_model
p4_va = final_model.predict_proba(Xva_np)[:, 1]
p4_te = final_model.predict_proba(Xu_te.to_numpy())[:, 1]
r4 = score_rung("L4", f"{SELECTED}, all features", len(FEATS), p4_va, p4_te)
ladder_tbl = show(ladder)
ladder_tbl.to_csv(T2_RES / "ladder_test.csv", index=False)
print("Baseline ladder, TEST (Se/PPV/Sp at the validation recall-0.80 threshold; L2 on score-available stays):")
display(ladder_tbl)

tn, fp, fn, tp = confusion_matrix(y_te, (p4_te >= r4["threshold"]).astype(int)).ravel()
print(f"L4 at threshold {r4['threshold']:.3f}: flags {tp + fp} of {len(y_te)} patients ({100 * (tp + fp) / len(y_te):.0f} %); "
      f"{tp} of {tp + fp} flagged die ({100 * tp / (tp + fp):.1f} % vs base rate {100 * y_te.mean():.1f} %); "
      f"misses {fn} of {tp + fn} deaths")
display(pd.DataFrame([[tp, fn], [fp, tn]], index=["died", "survived"], columns=["flagged", "not flagged"]))

In [ ]:
lt = pd.DataFrame(ladder)
lt["label"] = lt.rung + "  " + lt.model.str.replace(", all features", "").str.replace("logistic, ", "LR, ")
fig, axes_ = plt.subplots(1, 2, figsize=(11.5, 4.6), sharey=True)
ypos = np.arange(len(lt))[::-1]
for ax, metric, ref, ref_lbl in ((axes_[0], "AUROC", 0.5, "no skill 0.5"), (axes_[1], "AUPRC", y_te.mean(), "no skill = prevalence")):
    colors = [SERIES[0] if r == "L4" else (SERIES[1] if r.startswith("L3") else BASELINE) for r in lt.rung]
    ax.errorbar(lt[metric], ypos, xerr=[lt[metric] - lt[f"{metric} lo"], lt[f"{metric} hi"] - lt[metric]],
                fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
    ax.scatter(lt[metric], ypos, color=colors, s=60, zorder=3, edgecolor=SURFACE, linewidth=2)
    ax.axvline(ref, color=BASELINE, ls="--", lw=1)
    ax.set(xlabel=f"test {metric} (95% CI); dashed line = {ref_lbl}", title=f"{metric} by rung")
    ax.grid(axis="y", visible=False)
axes_[0].set_yticks(ypos); axes_[0].set_yticklabels(lt.label)
fig.tight_layout()
savefig(fig, "task2", "B5_ladder_test")

fig, axes_ = plt.subplots(1, 2, figsize=(10.5, 4.3))
for (name, s), color in zip((("L3", l3_predict(Xi_te)), ("L3-revised", l3r_predict(Xi_te)), ("L4 " + SELECTED, p4_te)), SERIES[:3]):
    fpr, tpr, _ = roc_curve(y_te, s); prec, rec, _ = precision_recall_curve(y_te, s)
    axes_[0].plot(fpr, tpr, color=color, label=f"{name} ({roc_auc_score(y_te, s):.3f})")
    axes_[1].plot(rec, prec, color=color, label=f"{name} ({average_precision_score(y_te, s):.3f})")
axes_[0].plot([0, 1], [0, 1], color=BASELINE, ls="--", lw=1, label="no skill")
axes_[1].axhline(y_te.mean(), color=BASELINE, ls="--", lw=1, label="no skill (prevalence)")
axes_[0].set(xlabel="false-positive rate", ylabel="true-positive rate", title="ROC, test (AUROC)")
axes_[1].set(xlabel="recall", ylabel="precision", title="Precision-recall, test (AUPRC)")
for ax in axes_:
    ax.legend(fontsize=8.5)
fig.tight_layout()
savefig(fig, "task2", "B5_roc_pr_test")

In [ ]:
h2h = {}
for score, key in (("SAPS-I", "saps_i"), ("SOFA", "sofa")):
    mt = (te_lab[f"{key}_available"] == 1).to_numpy()
    yb = y_te[mt]
    b = strat_boot(yb, te_lab.loc[mt, key].to_numpy(), roc_auc_score)
    l4 = strat_boot(yb, p4_te[mt], roc_auc_score)
    h2h[f"vs {score} (n = {mt.sum():,})"] = {"bedside AUROC [CI]": ci_str(*b), "L4 AUROC [CI]": ci_str(*l4),
                                             "gain": l4[0] - b[0], "CIs overlap": not (l4[1] > b[2] or l4[2] < b[1])}
print("Head-to-head on each bedside score's own patients (test):")
pd.DataFrame(h2h).T

## B6 Calibration and subgroups (frozen test predictions)

These analyses use the test prediction vector that was already scored in B5. They involve no retraining and no new choices, so they are not a second look.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_te, p4_te, n_bins=10, strategy="uniform")
print(f"Brier score (test): {brier_score_loss(y_te, p4_te):.4f}  (constant-prevalence predictor: {y_te.mean() * (1 - y_te.mean()):.4f})")
display(pd.DataFrame({"mean predicted": mean_pred, "observed death rate": frac_pos}).round(3))

icu_names = {1: "Coronary care", 2: "Cardiac surgery recovery", 3: "Medical", 4: "Surgical"}
sg = []
for k, name in icu_names.items():
    msk = Xu_te[f"static_icutype_{k}"].to_numpy() == 1
    auc = strat_boot(y_te[msk], p4_te[msk], roc_auc_score)
    sg.append({"ICU type": name, "n": int(msk.sum()), "deaths": int(y_te[msk].sum()), "prevalence": y_te[msk].mean(),
               "AUROC": auc[0], "lo": auc[1], "hi": auc[2]})
sg = pd.DataFrame(sg)
display(sg.round(3))

fig, axes_ = plt.subplots(1, 2, figsize=(10.5, 4.2), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axes_[0]
ax.plot([0, 1], [0, 1], color=BASELINE, ls="--", lw=1, label="perfect calibration")
ax.plot(mean_pred, frac_pos, color=SERIES[0], marker="s", label=f"L4 {SELECTED} (Brier {brier_score_loss(y_te, p4_te):.3f})")
ax.set(xlabel="mean predicted probability", ylabel="observed death rate", title="Calibration, test", xlim=(0, 1), ylim=(0, 1))
ax.legend(fontsize=9, loc="upper left")
ax = axes_[1]
ypos = np.arange(len(sg))[::-1]
ax.errorbar(sg.AUROC, ypos, xerr=[sg.AUROC - sg.lo, sg.hi - sg.AUROC], fmt="none", ecolor=INK2, elinewidth=1, capsize=3)
ax.scatter(sg.AUROC, ypos, color=SERIES[0], s=60, zorder=3, edgecolor=SURFACE, linewidth=2)
ax.axvline(r4["AUROC"], color=BASELINE, ls="--", lw=1)
ax.set(yticks=ypos, yticklabels=[f"{r['ICU type']} (n={r.n}, {r.deaths} deaths)" for _, r in sg.iterrows()],
       xlabel=f"test AUROC (95% CI); dashed line = overall {r4['AUROC']:.3f}", title="Subgroups by ICU type")
ax.grid(axis="y", visible=False)
fig.tight_layout()
savefig(fig, "task2", "B6_calibration_subgroups_test")

## B7 Interpretation: what the model learned

SHAP values come from XGBoost's exact TreeSHAP (`pred_contribs`) on the test set, the frozen prediction vector. We also check whether CatBoost, trained independently, agrees on what matters, computing SHAP on *validation* because CatBoost lost model selection and must never be scored on test.

In [ ]:
Xte_np = Xu_te.to_numpy()
contrib_te = xgb_model.get_booster().predict(xgb.DMatrix(Xte_np), pred_contribs=True)
shap_te, base_te = contrib_te[:, :-1], contrib_te[:, -1]


def stat_type(n):
    for key, lab_ in (("measured", "measurement behaviour"), ("__count", "measurement behaviour"),
                      ("time_span_hours", "measurement behaviour"), ("slope_per_hour", "trend"), ("__delta", "trend"),
                      ("__std", "variability"), ("__first", "point in time"), ("__last", "point in time"),
                      ("__min", "extreme"), ("__max", "extreme"), ("__mean", "central"), ("__median", "central"),
                      ("__total", "cumulative")):
        if key in n:
            return lab_
    return "static"


def organ(n):
    n = n.lower()
    for keys, lab_ in ((["bun", "creatinine", "urine"], "renal"), (["gcs"], "CNS"),
                       (["lactate", "bp_", "sysabp", "diasabp", "map", "hr__"], "cardiovascular"),
                       (["platelets", "wbc", "hct"], "haematology"), (["bilirubin", "alt", "ast", "alp", "albumin"], "hepatic"),
                       (["pao2", "paco2", "fio2", "sao2", "mechvent", "resprate"], "respiratory"),
                       (["age", "gender", "icutype", "bmi", "height", "weight"], "demographic / context"),
                       (["record__"], "record-level")):
        if any(k in n for k in keys):
            return lab_
    return "other lab"


mean_abs = pd.Series(np.abs(shap_te).mean(0), index=FEATS).sort_values(ascending=False)
top20 = mean_abs.head(20).rename("mean |SHAP|").to_frame()
top20["organ"] = [organ(f) for f in top20.index]
top20["statistic type"] = [stat_type(f) for f in top20.index]
top20["in L3 / L3-revised"] = [("L3 " if f in L3_FEATURES else "") + ("L3r" if f in L3R_FEATURES else "") for f in top20.index]
top20.to_csv(T2_RES / "L4_shap_top20_test.csv")
display(top20.round(3))
print("statistic-type counts in the top 20:", top20["statistic type"].value_counts().to_dict())

shap.summary_plot(shap_te, Xte_np, feature_names=FEATS, max_display=20, show=False, plot_size=(8, 7))
plt.title("SHAP summary, L4 XGBoost (test)", loc="left", fontsize=11, fontweight="bold")
savefig(plt.gcf(), "task2", "B7_shap_summary_test")

In [ ]:
sx = xgb_model.get_booster().predict(xgb.DMatrix(Xva_np), pred_contribs=True)[:, :-1]
sc_ = cb_model.get_feature_importance(Pool(Xva_np), type="ShapValues")[:, :-1]
rank_x = pd.Series(np.abs(sx).mean(0), index=FEATS).sort_values(ascending=False)
rank_c = pd.Series(np.abs(sc_).mean(0), index=FEATS).sort_values(ascending=False)
side = pd.DataFrame({"XGBoost feature": rank_x.index[:15], "XGBoost |SHAP|": rank_x.values[:15],
                     "CatBoost feature": rank_c.index[:15], "CatBoost |SHAP|": rank_c.values[:15]}, index=range(1, 16))
display(side.round(3))
overlap = set(rank_x.index[:20]) & set(rank_c.index[:20])
print(f"top-20 overlap between independently trained models (validation SHAP): {len(overlap)}/20")

### Three patients (individual explanations)

We pick three cases from the frozen test predictions: the most confident correct high-risk call, the most confident false alarm, and the most confident missed death. The waterfalls show how each feature moves the log-odds away from the average prediction.

In [ ]:
p_test = p4_te if SELECTED == "XGBoost" else xgb_model.predict_proba(Xte_np)[:, 1]
cand = {
    "TP_high_risk_died": np.where((p_test > 0.7) & (y_te == 1))[0], "FP_high_risk_survived": np.where((p_test > 0.5) & (y_te == 0))[0],
    "FN_low_risk_died": np.where((p_test < 0.3) & (y_te == 1))[0]}
cases = {"TP_high_risk_died": cand["TP_high_risk_died"][np.argmax(p_test[cand["TP_high_risk_died"]])],
         "FP_high_risk_survived": cand["FP_high_risk_survived"][np.argmax(p_test[cand["FP_high_risk_survived"]])],
         "FN_low_risk_died": cand["FN_low_risk_died"][np.argmin(p_test[cand["FN_low_risk_died"]])]}
case_rows = []
for label, i in cases.items():
    exp = shap.Explanation(values=shap_te[i], base_values=base_te[i], data=Xte_np[i], feature_names=FEATS)
    shap.plots.waterfall(exp, max_display=12, show=False)
    fig = plt.gcf(); fig.set_size_inches(8, 5.2)
    plt.title(f"{label.replace('_', ' ')}: RecordID {ids_te[i]}, predicted {p_test[i]:.3f}, outcome {y_te[i]}",
              loc="left", fontsize=10, fontweight="bold")
    savefig(fig, "task2", f"B7_shap_waterfall_{label}")
    top = np.argsort(-np.abs(shap_te[i]))[:6]
    case_rows += [{"case": label, "RecordID": ids_te[i], "predicted": p_test[i], "feature": FEATS[j],
                   "value": Xte_np[i, j], "SHAP (log-odds)": shap_te[i, j]} for j in top]
pd.DataFrame(case_rows).round(3)

## B8 The 2012 challenge metric (validation only)

The 2012 challenge scored entries on min(Se, PPV). Our operating point was chosen for recall ≈ 0.80, on clinical grounds, which does not maximise that metric. Here we sweep the threshold on **validation only**; test is not re-thresholded, because that would be a second look.

In [ ]:
thr_grid = np.linspace(0.01, 0.99, 197)
mse = np.array([min_se_ppv(y_va, (p4_va >= t).astype(int)) for t in thr_grid])
t_best = thr_grid[mse.argmax()]
m_best, m_rec = compute_metrics(y_va, p4_va, threshold=t_best), compute_metrics(y_va, p4_va, threshold=r4["threshold"])
pd.DataFrame({"recall-0.80 point (ours)": m_rec, "min(Se,PPV)-optimal point": m_best}).T.assign(
    threshold=[r4["threshold"], t_best])[["threshold", "sensitivity", "ppv", "specificity", "min_se_ppv"]].round(3)

For reference, on the challenge's own hidden test sets (B + C) the SAPS-I benchmark scored 0.313 and the winning entry 0.535. Those are different test sets, so the comparison is approximate.

At the validation-optimal threshold of 0.345, min(Se, PPV) is 0.518, which would have been competitive. But it halves sensitivity, from 0.80 to 0.52, and that is the trade this project's clinical framing rejects: a missed death costs more than an extra review.

## B9 Discussion

**Headline.** At 48 h, XGBoost on all 1,474 features predicts in-hospital death with test **AUROC 0.885 [0.865, 0.903]** and **AUPRC 0.591 [0.535, 0.649]**, about 4× the prevalence of 0.144. On their own patients, the bedside scores reach AUROC 0.640 (SAPS-I) and 0.623 (SOFA), with CIs disjoint from L4's.

**Reading the ladder.**

* **One lab beats both bedside scores.** BUN max alone (0.675) matches or beats them. SAPS-I and SOFA were built for population stratification and take only a few dozen distinct values.
* **A summary statistic is worth 0.079 AUROC.** L3 and L3-revised share the same parameters and recipe; swapping two summary statistics takes AUROC from 0.742 to 0.821.
* **The full model adds a further +0.064 AUROC and +0.091 AUPRC** over L3-revised, with disjoint AUROC CIs.

**Three findings carry the discussion.**

1. **Feature breadth, not non-linearity, does most of the work.** On its top 8 features XGBoost reaches validation AUROC 0.815, within noise of the 8-feature logistic regression (0.808). At 50 features it reaches 0.859, 99.5 % of the full model's validation AUROC.
2. **Which summary of a variable you take matters as much as which variable.** Last GCS (AUROC 0.752 on train + validation) beats minimum GCS (0.588) by a wide margin. Almost every ICU patient records a low GCS at some point from sedation, intubation or procedures, so the minimum saturates; the last value reads consciousness after those effects wear off. The clinical prior picked the right variable and the wrong summary of it.
3. **The model generalises and ranks well, but is over-confident at the top.**
   * *Fit vs generalisation.* XGBoost fits the training set exactly (train AUROC 1.0) yet holds 0.866 on validation and 0.885 on test. CatBoost, which early-stops at 224 trees, shows a similar gap, and adding early stopping to XGBoost changes validation AUROC by 0.001. That is the capacity to interpolate, not degraded generalisation.
   * *Calibration.* The Brier score is 0.086, against 0.123 for a constant predictor. The reliability curve tracks the diagonal up to about 0.35, then sits below it: predicted 0.65 observes 0.54, and predicted 0.95 observes 0.83. This is consistent with `scale_pos_weight` inflating scores.
   * *Subgroups.* AUROC ranges from 0.837 (medical ICU) to 0.955 (cardiac-surgery recovery, only 19 deaths, widest CI). No ICU type falls below 0.83.

**What the model uses.**

* **GCS dominates.** Last GCS has 2.7× the importance of age, the next feature, and 7 of the top 20 features are GCS statistics.
* **Renal output and trends.** Four are renal (urine totals and BUN), and two are trends (white-cell count and GCS slopes).
* **Measurement behaviour.** Two are measurement-behaviour features: the time span of ventilation and of haematocrit charting.
* **Not an artefact of one implementation.** CatBoost, trained independently, puts the same three features first and shares 12 of the top 20.

**The three patients:**

* **Correct high-risk call (103032).** Every organ system points the same way: last GCS 7, lactate 11.6 mmol/L, second-day urine 414 mL, platelets 25.
* **False alarm (100571).** Driven by a last GCS of 6 (+1.74 log-odds) in an 80-year-old with otherwise adequate perfusion. Most likely sedation read as prognosis, which is the failure mode of the last-GCS feature.
* **Missed death (102169).** Looked like a recovering patient at 48 h: 7.2 L urine, lactate 0.6, age 53. Deaths after an apparently benign first 48 h are a residual that no 48-hour model will catch.

**At the operating point**, the model flags 519 of 1,775 patients (29 %). Of those, 38.7 % die, a 2.7-fold enrichment over the base rate, and it misses 55 of 256 deaths. To reach the same sensitivity, SAPS-I needs a specificity of 0.35, i.e. it flags about two-thirds of patients.

**Limitations.**

1. **Internal validation only.** One cohort, one split. Last GCS depends on charting cadence and needs checking at another site.
2. **Over-confident probabilities above ~0.35.** Recalibrate on validation (isotonic or Platt) before quoting risks to families.
3. **Two counter-intuitive L3 coefficients.** Creatinine and platelets have unexpected signs, from collinearity with BUN. They are left as they are, because changing the feature set would be another test look.
4. **A benchmark, not a deployment model.** The 1,474-feature model is a research benchmark; a 50–200 feature version keeps essentially all the signal and is far easier to audit.
5. **The outcome includes post-ICU deaths.** In-hospital death counts deaths after ICU discharge.
6. **Fairness not assessed.** Fairness by sex or age band was not checked, though age is the second most important feature.
7. **Pre-processing side effects.** The age range check blanks the 19 stays aged 15–17, whose age is then imputed. 167 stays with negative length of stay are excluded.

---
# Part C · Summary and saved artefacts

| | Task 1: SOFA forecast (6 h ahead) | Task 2: in-hospital mortality at 48 h |
|---|---|---|
| Best model | ordinal decomposition (HistGB threshold classifiers) | XGBoost, 1,474 features |
| Test result | total-SOFA MAE 0.591 vs persistence 0.726 (mean of 4 horizons) | AUROC 0.885 [0.865, 0.903], AUPRC 0.591 [0.535, 0.649] |
| Simple baseline | persistence; beaten at 4/4 horizons by 4 of 5 families | SAPS-I 0.640, SOFA 0.623; L3-revised logistic 0.821 |
| Secondary result | deterioration AUROC 0.838, top 10 % of checkpoints catch 49 % of events | 50 features keep 99.5 % of validation AUROC |

**Themes the two tasks share:**

* **The baseline decides the story.** For SOFA, "nothing changes" is hard to beat except where the score genuinely moves. For mortality, one lab value already beats the bedside scores.
* **Consciousness carries the signal.** The neurological organ is the only one where forecasting beats persistence. The last recorded GCS is by far the strongest mortality predictor.
* **Measurement behaviour is information.** Hours since the last measurement (task 1) and charting time spans (task 2) rank among the important features. Missing values are therefore kept and flagged rather than imputed away.
* **Ranking is good; probabilities are not yet trustworthy.** Both classifiers are over-confident at the high end and need recalibration before their outputs are read as risks.
* **Shared limits.** One public dataset, internal validation only, and no medication data.

**Cohorts and splits differ.** The two tasks use different cohorts (12,000 vs 11,833 stays) and independent splits (seeds 42 vs 20260907). Each task's model selection and test evaluation are self-contained, so results should not be compared row-by-row across tasks.

The cell below lists every artefact this notebook reads or writes:

* `preprocessed/`: pre-processing outputs.
* `trained_models/`: fitted models and the task 1 model specification.
* `results/`: figures and result tables.

In [ ]:
def tree(path, depth=0, max_files=8):
    files = sorted(p for p in path.iterdir() if p.is_file())
    for p in files[:max_files]:
        print("    " * depth + f"{p.name}  ({p.stat().st_size / 1e6:.2f} MB)")
    if len(files) > max_files:
        print("    " * depth + f"... and {len(files) - max_files} more files")
    for d in sorted(p for p in path.iterdir() if p.is_dir()):
        print("    " * depth + f"{d.name}/")
        tree(d, depth + 1, max_files)


for top in ("preprocessed", "trained_models", "results"):
    print(f"{top}/"); tree(ROOT / top, 1)